# Recommendation Baselines — Phase 3

This notebook implements the complete Phase 3 recommendation-baseline workflow (Steps 1-16): contract validation, leakage-safe model construction and discovery-only selection, final validation evaluation, artifact consolidation and decision rules for Phase 4.

Validation-future data is evaluation-only. Discovery-future labels may be used for model training in later steps.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Step 1 — Imports, paths and immutable experiment configuration

In [2]:
from pathlib import Path
import json
import time

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

BENCHMARK_ROOT = Path('/content/drive/MyDrive/datasets/recommendation_benchmark_final_outputs')
OUTPUT_ROOT = Path('/content/drive/MyDrive/datasets/recommendation_baseline_outputs')
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

DATASET_DIRS = {
    'ASSISTments': BENCHMARK_ROOT / 'assistments',
    'KDD': BENCHMARK_ROOT / 'kdd',
}
METRIC_KS = (5, 10, 20)
PRIMARY_RELEVANCE = 'attempted'
RANDOM_STATE = 42

ROOT_FILES = [
    'benchmark_summary.csv', 'artifact_manifest.csv',
    'benchmark_config.json', 'benchmark_schema.json',
]
for filename in ROOT_FILES:
    path = BENCHMARK_ROOT / filename


## Step 2 — Validate the 18-artifact benchmark contract

This validation uses Parquet metadata wherever possible so the four-million-row ASSISTments histories are not loaded merely to check their schemas. Small learner and catalogue tables are loaded for semantic assertions.

In [3]:
with open(BENCHMARK_ROOT / 'benchmark_config.json', encoding='utf-8') as file:
    benchmark_config = json.load(file)
with open(BENCHMARK_ROOT / 'benchmark_schema.json', encoding='utf-8') as file:
    benchmark_schema = json.load(file)
benchmark_summary = pd.read_csv(BENCHMARK_ROOT / 'benchmark_summary.csv')
phase2_manifest = pd.read_csv(BENCHMARK_ROOT / 'artifact_manifest.csv')

assert benchmark_config['candidate_statistics_source'] == 'discovery_early_only'
assert benchmark_config['future_role'] == 'relevance_labels_only'
assert benchmark_config['assist_skill_name_mapping_source'] == 'discovery_early_only'
assert len(benchmark_schema) == 9
assert len(phase2_manifest) == 18
assert set(phase2_manifest['Dataset']) == set(DATASET_DIRS)

REQUIRED_COLUMNS = {
    'learner_splits.parquet': {
        'learner_id', 'cohort', 'evaluable_skill', 'evaluable_problem',
        'cold_start_skill_history', 'cold_start_problem_history',
        'cluster', 'cluster_selection_status', 'cluster_allowed_as_primary',
    },
    'skill_catalog.parquet': {
        'item_id', 'item_type', 'training_interactions', 'training_learners',
        'training_success_rate', 'training_popularity', 'catalog_source',
    },
    'problem_catalog.parquet': {
        'item_id', 'item_type', 'training_interactions', 'training_learners',
        'training_success_rate', 'training_popularity', 'catalog_source',
        'problem_type', 'hierarchy',
    },
    'early_skill_history.parquet': {
        'learner_id', 'item_id', 'early_interaction_count',
        'empirical_bayes_mastery', 'mastery_evidence_confidence',
        'skill_state', 'in_candidate_catalog',
    },
    'future_skill_relevance.parquet': {
        'learner_id', 'item_id', 'relevance_binary',
        'successful_future_item', 'in_candidate_catalog', 'seen_in_early',
    },
    'early_problem_history.parquet': {
        'learner_id', 'item_id', 'early_interaction_count',
        'early_success_rate', 'in_candidate_catalog',
    },
    'future_problem_relevance.parquet': {
        'learner_id', 'item_id', 'relevance_binary',
        'successful_future_item', 'in_candidate_catalog', 'seen_in_early',
    },
    'problem_skill_map.parquet': {
        'problem_item_id', 'skill_item_id', 'training_interactions',
        'association_share', 'mapping_source',
    },
    'skill_name_id_map.parquet': {
        'normalised_skill_name', 'mapped_skill_id', 'training_rows',
        'training_learners', 'distinct_skill_ids', 'mapping_source',
    },
}
assert set(REQUIRED_COLUMNS) == set(benchmark_schema)

contract_rows = []
small_tables = {}
for dataset, directory in DATASET_DIRS.items():
    dataset_manifest = phase2_manifest[phase2_manifest['Dataset'].eq(dataset)].set_index('File')
    assert set(dataset_manifest.index) == set(REQUIRED_COLUMNS)
    for filename, required_columns in REQUIRED_COLUMNS.items():
        path = directory / filename
        parquet_file = pq.ParquetFile(path)
        row_count = parquet_file.metadata.num_rows
        schema_columns = set(parquet_file.schema_arrow.names)
        missing_columns = sorted(required_columns - schema_columns)
        manifest_row = dataset_manifest.loc[filename]
        assert row_count == int(manifest_row['Rows'])
        assert path.stat().st_size == int(manifest_row['Bytes'])
        assert not missing_columns, f'{dataset}/{filename}: {missing_columns}'
        contract_rows.append({
            'Dataset': dataset, 'File': filename, 'Rows': row_count,
            'Bytes': path.stat().st_size, 'SchemaValid': True,
            'ManifestValid': True,
        })

    learners = pd.read_parquet(directory / 'learner_splits.parquet')
    skill_catalog = pd.read_parquet(directory / 'skill_catalog.parquet')
    problem_catalog = pd.read_parquet(directory / 'problem_catalog.parquet')
    problem_skill_map = pd.read_parquet(directory / 'problem_skill_map.parquet')
    skill_name_map = pd.read_parquet(directory / 'skill_name_id_map.parquet')
    assert set(learners['cohort']) == {'discovery', 'validation'}
    assert learners['cluster'].notna().all()
    assert skill_catalog['catalog_source'].eq('discovery_early_only').all()
    assert problem_catalog['catalog_source'].eq('discovery_early_only').all()
    assert problem_skill_map['mapping_source'].eq('discovery_early_only').all()
    if len(skill_name_map):
        assert skill_name_map['mapping_source'].eq('discovery_early_only').all()
    small_tables[dataset] = {
        'learners': learners, 'skill_catalog': skill_catalog,
        'problem_catalog': problem_catalog,
    }

assert small_tables['ASSISTments']['learners']['cluster_selection_status'].eq('accepted').all()
assert small_tables['KDD']['learners']['cluster_selection_status'].eq('exploratory_fallback').all()
assert not small_tables['KDD']['learners']['cluster_allowed_as_primary'].any()
contract_validation = pd.DataFrame(contract_rows)
display(benchmark_summary)
display(contract_validation)
print('Validated 18 Phase 2 Parquet artifacts without loading the large histories.')

,Dataset,RawRows,RawLearners,EligibleLearners,DiscoveryLearners,ValidationLearners,EarlyInteractions,FutureInteractions,SkillCandidates,ProblemCandidates,...,ValidationProblemEvaluableRate,ValidationSkillColdStartRate,ValidationProblemColdStartRate,ClusterSelectionStatus,TrainingSkillNameMappings,AmbiguousTrainingSkillNames,EarlyRowsMappedFromName,FutureRowsMappedFromName,EarlyTextFallbackRows,FutureTextFallbackRows
0,ASSISTments,6117947,46667,33335,26668,6667,4184236,1814850,161,39779,...,0.899355,0.375731,0.024599,accepted,194,33,0,0,0,0
1,KDD,809694,574,565,452,113,566460,243152,99,855,...,0.982301,0.000000,0.000000,exploratory_fallback,0,0,0,0,0,0


,Dataset,File,Rows,Bytes,SchemaValid,ManifestValid
0,ASSISTments,learner_splits.parquet,33335,1283461,True,True
1,ASSISTments,skill_catalog.parquet,161,16637,True,True
2,ASSISTments,problem_catalog.parquet,39779,978772,True,True
3,ASSISTments,early_skill_history.parquet,260768,8707997,True,True
4,ASSISTments,future_skill_relevance.parquet,174823,4960293,True,True
5,ASSISTments,early_problem_history.parquet,4052498,76911901,True,True
6,ASSISTments,future_problem_relevance.parquet,1783782,35311927,True,True
7,ASSISTments,problem_skill_map.parquet,17826,176578,True,True
8,ASSISTments,skill_name_id_map.parquet,194,10524,True,True
9,KDD,learner_splits.parquet,565,46435,True,True


Validated 18 Phase 2 Parquet artifacts without loading the large histories.


## Step 3 — Define datasets, tasks and relevance labels

Later attempted items are the primary offline target. Later successful items are a sensitivity target.

In [4]:
RELEVANCE_DEFINITIONS = {
    'attempted': 'relevance_binary',
    'successful': 'successful_future_item',
}
TASK_DEFINITIONS = [
    {
        'Dataset': 'ASSISTments', 'Task': 'problem', 'Priority': 'primary',
        'CatalogFile': 'problem_catalog.parquet',
        'EarlyHistoryFile': 'early_problem_history.parquet',
        'FutureRelevanceFile': 'future_problem_relevance.parquet',
        'EvaluableColumn': 'evaluable_problem',
        'ColdStartColumn': 'cold_start_problem_history',
        'ClusterUse': 'accepted_ablation',
    },
    {
        'Dataset': 'ASSISTments', 'Task': 'skill', 'Priority': 'secondary',
        'CatalogFile': 'skill_catalog.parquet',
        'EarlyHistoryFile': 'early_skill_history.parquet',
        'FutureRelevanceFile': 'future_skill_relevance.parquet',
        'EvaluableColumn': 'evaluable_skill',
        'ColdStartColumn': 'cold_start_skill_history',
        'ClusterUse': 'accepted_ablation',
    },
    {
        'Dataset': 'KDD', 'Task': 'problem', 'Priority': 'external_validation',
        'CatalogFile': 'problem_catalog.parquet',
        'EarlyHistoryFile': 'early_problem_history.parquet',
        'FutureRelevanceFile': 'future_problem_relevance.parquet',
        'EvaluableColumn': 'evaluable_problem',
        'ColdStartColumn': 'cold_start_problem_history',
        'ClusterUse': 'exploratory_ablation_only',
    },
    {
        'Dataset': 'KDD', 'Task': 'skill', 'Priority': 'external_validation',
        'CatalogFile': 'skill_catalog.parquet',
        'EarlyHistoryFile': 'early_skill_history.parquet',
        'FutureRelevanceFile': 'future_skill_relevance.parquet',
        'EvaluableColumn': 'evaluable_skill',
        'ColdStartColumn': 'cold_start_skill_history',
        'ClusterUse': 'exploratory_ablation_only',
    },
]
task_table = pd.DataFrame(TASK_DEFINITIONS)
summary_lookup = benchmark_summary.set_index('Dataset')
task_table['CandidateCount'] = task_table.apply(
    lambda row: int(summary_lookup.loc[row['Dataset'], 'ProblemCandidates' if row['Task'] == 'problem' else 'SkillCandidates']),
    axis=1,
)
task_table['ValidationEvaluableRate'] = task_table.apply(
    lambda row: float(summary_lookup.loc[row['Dataset'], 'ValidationProblemEvaluableRate' if row['Task'] == 'problem' else 'ValidationSkillEvaluableRate']),
    axis=1,
)
task_table['ValidationColdStartRate'] = task_table.apply(
    lambda row: float(summary_lookup.loc[row['Dataset'], 'ValidationProblemColdStartRate' if row['Task'] == 'problem' else 'ValidationSkillColdStartRate']),
    axis=1,
)
assert len(task_table) == 4
assert task_table['CandidateCount'].gt(0).all()
assert set(RELEVANCE_DEFINITIONS) == {'attempted', 'successful'}
display(task_table)

,Dataset,Task,Priority,CatalogFile,EarlyHistoryFile,FutureRelevanceFile,EvaluableColumn,ColdStartColumn,ClusterUse,CandidateCount,ValidationEvaluableRate,ValidationColdStartRate
0,ASSISTments,problem,primary,problem_catalog.parquet,early_problem_history.parquet,future_problem_relevance.parquet,evaluable_problem,cold_start_problem_history,accepted_ablation,39779,0.899355,0.024599
1,ASSISTments,skill,secondary,skill_catalog.parquet,early_skill_history.parquet,future_skill_relevance.parquet,evaluable_skill,cold_start_skill_history,accepted_ablation,161,0.588271,0.375731
2,KDD,problem,external_validation,problem_catalog.parquet,early_problem_history.parquet,future_problem_relevance.parquet,evaluable_problem,cold_start_problem_history,exploratory_ablation_only,855,0.982301,0.000000
3,KDD,skill,external_validation,skill_catalog.parquet,early_skill_history.parquet,future_skill_relevance.parquet,evaluable_skill,cold_start_skill_history,exploratory_ablation_only,99,0.991150,0.000000


## Step 4 — Candidate policies

Candidate-policy filtering operates on already-generated sparse score rows.

In [5]:
CANDIDATE_POLICIES = {
    'all_supported': {
        'exclude_seen': False,
        'description': 'Allow supported items seen previously; repeated practice is valid.',
    },
    'novel_only': {
        'exclude_seen': True,
        'description': 'Exclude items present in the learner early history.',
    },
}

def apply_candidate_policy(scored_candidates, early_history, policy):
    if policy not in CANDIDATE_POLICIES:
        raise ValueError(f'Unknown candidate policy: {policy}')
    required = {'learner_id', 'item_id', 'score'}
    if not required.issubset(scored_candidates.columns):
        raise ValueError(f'Scored candidates require columns: {sorted(required)}')
    candidates = scored_candidates.copy()
    if not CANDIDATE_POLICIES[policy]['exclude_seen']:
        return candidates
    seen = (
        early_history.loc[early_history['in_candidate_catalog'], ['learner_id', 'item_id']]
        .drop_duplicates().assign(_seen=True)
    )
    candidates = candidates.merge(seen, on=['learner_id', 'item_id'], how='left')
    return candidates[candidates['_seen'].ne(True)].drop(columns='_seen')

def assert_candidate_subset(scored_candidates, catalog):
    candidate_items = set(scored_candidates['item_id'])
    catalog_items = set(catalog['item_id'])
    unknown = candidate_items - catalog_items
    assert not unknown, f'{len(unknown)} scored items are outside the candidate catalogue.'


## Step 5 — Ranking, coverage and cold-start metrics

Only learners with at least one in-catalog relevant future item are evaluable. Incomplete recommendation lists are penalised using K as the Precision@K denominator.

In [6]:
def metrics_for_user(recommended_items, relevant_items, k):
    recommended = list(recommended_items[:k])
    relevant = set(relevant_items)
    if not relevant:
        return None
    hits = np.array([item in relevant for item in recommended], dtype=float)
    padded_hits = np.pad(hits, (0, max(0, k - len(hits))))[:k]
    hit_count = padded_hits.sum()
    precision = hit_count / k
    recall = hit_count / len(relevant)
    discounts = np.log2(np.arange(2, k + 2))
    dcg = np.sum(padded_hits / discounts)
    ideal_length = min(len(relevant), k)
    idcg = np.sum(np.ones(ideal_length) / discounts[:ideal_length])
    ndcg = dcg / idcg if idcg else 0.0
    hit_positions = np.flatnonzero(padded_hits)
    average_precision = (
        sum(padded_hits[:position + 1].sum() / (position + 1) for position in hit_positions)
        / min(len(relevant), k)
    )
    return {
        'Precision': precision, 'Recall': recall, 'NDCG': ndcg,
        'MAP': average_precision, 'HitRate': float(hit_count > 0),
    }

def evaluate_recommendations(
    recommendations, relevance, catalog, learner_splits,
    relevance_column, evaluable_column, cold_start_column,
    dataset, task, candidate_policy, relevance_name, ks=METRIC_KS,
    evaluation_learner_ids=None,
):
    started_at = time.perf_counter()
    required_recommendation_columns = {'learner_id', 'item_id', 'score'}
    if not required_recommendation_columns.issubset(recommendations.columns):
        raise ValueError('Recommendations require learner_id, item_id and score.')
    if candidate_policy not in CANDIDATE_POLICIES:
        raise ValueError(f'Unknown candidate policy: {candidate_policy}')
    if relevance_column not in relevance.columns:
        raise ValueError(f'Missing relevance column: {relevance_column}')
    assert not recommendations.duplicated(['learner_id', 'item_id']).any()
    assert np.isfinite(pd.to_numeric(recommendations['score'], errors='coerce')).all()
    assert_candidate_subset(recommendations, catalog)

    ranked = recommendations.sort_values(
        ['learner_id', 'score', 'item_id'], ascending=[True, False, True],
        kind='mergesort',
    )
    relevance_mask = (
        relevance['in_candidate_catalog'] & relevance[relevance_column].eq(1)
    )
    if candidate_policy == 'novel_only':
        if 'seen_in_early' not in relevance.columns:
            raise ValueError('novel_only evaluation requires seen_in_early labels.')
        relevance_mask &= ~relevance['seen_in_early'].fillna(False).astype(bool)
    relevant = relevance.loc[
        relevance_mask, ['learner_id', 'item_id']
    ].drop_duplicates()
    relevant_by_user = relevant.groupby('learner_id')['item_id'].agg(set).to_dict()
    ranked_by_user = ranked.groupby('learner_id')['item_id'].agg(list).to_dict()

    if evaluation_learner_ids is None:
        evaluation_learners = learner_splits[
            learner_splits['cohort'].eq('validation')
        ].copy()
        evaluation_scope = 'validation'
    else:
        evaluation_ids = set(pd.Series(evaluation_learner_ids).astype(str))
        evaluation_learners = learner_splits[
            learner_splits['learner_id'].astype(str).isin(evaluation_ids)
        ].copy()
        assert set(evaluation_learners['learner_id'].astype(str)) == evaluation_ids
        evaluation_scope = 'provided_learner_ids'
    eligible_users = set(evaluation_learners.loc[
        evaluation_learners[evaluable_column], 'learner_id'
    ])
    eligible_users &= set(relevant_by_user)
    segments = {
        'all': eligible_users,
        'cold_start': eligible_users & set(evaluation_learners.loc[evaluation_learners[cold_start_column], 'learner_id']),
        'non_cold_start': eligible_users & set(evaluation_learners.loc[~evaluation_learners[cold_start_column], 'learner_id']),
    }

    rows = []
    catalog_size = len(catalog)
    for segment_name, users in segments.items():
        if not users:
            continue
        for k in ks:
            user_rows = []
            recommended_union = set()
            list_lengths = []
            for learner_id in sorted(users):
                recommended = ranked_by_user.get(learner_id, [])[:k]
                values = metrics_for_user(recommended, relevant_by_user[learner_id], k)
                user_rows.append(values)
                recommended_union.update(recommended)
                list_lengths.append(len(recommended))
            user_metrics = pd.DataFrame(user_rows)
            rows.append({
                'Segment': segment_name, 'K': k,
                'PrecisionAtK': user_metrics['Precision'].mean(),
                'RecallAtK': user_metrics['Recall'].mean(),
                'NDCGAtK': user_metrics['NDCG'].mean(),
                'MAPAtK': user_metrics['MAP'].mean(),
                'HitRateAtK': user_metrics['HitRate'].mean(),
                'CatalogCoverageAtK': len(recommended_union) / catalog_size,
                'MeanRecommendations': float(np.mean(list_lengths)),
                'EvaluatedLearners': len(users),
                'EvaluableRate': len(eligible_users) / len(evaluation_learners),
                'EvaluationScope': evaluation_scope,
            })
    result = pd.DataFrame(rows)
    result.insert(0, 'RelevanceDefinition', relevance_name)
    result.insert(0, 'CandidatePolicy', candidate_policy)
    result.insert(0, 'Task', task)
    result.insert(0, 'Dataset', dataset)
    result['RecommendationMemoryBytes'] = int(recommendations.memory_usage(deep=True).sum())
    result['MetricRuntimeSeconds'] = time.perf_counter() - started_at
    return result

## Save and verify the Phase 3 steps 1-5 setup

In [7]:
contract_validation.to_csv(OUTPUT_ROOT / 'phase3_contract_validation.csv', index=False)
task_table.to_csv(OUTPUT_ROOT / 'phase3_task_definitions.csv', index=False)
phase3_setup_config = {
    'implemented_steps': [1, 2, 3, 4, 5],
    'pending_steps': list(range(6, 17)),
    'benchmark_root': str(BENCHMARK_ROOT),
    'output_root': str(OUTPUT_ROOT),
    'metric_ks': list(METRIC_KS),
    'primary_relevance': PRIMARY_RELEVANCE,
    'relevance_definitions': RELEVANCE_DEFINITIONS,
    'candidate_policies': CANDIDATE_POLICIES,
    'primary_task': 'ASSISTments problem recommendation',
    'dense_cross_product_forbidden': True,
    'validation_future_role': 'evaluation_only',
    'random_state': RANDOM_STATE,
}
with open(OUTPUT_ROOT / 'phase3_setup_config.json', 'w', encoding='utf-8') as file:
    json.dump(phase3_setup_config, file, indent=2)

setup_files = [
    OUTPUT_ROOT / 'phase3_contract_validation.csv',
    OUTPUT_ROOT / 'phase3_task_definitions.csv',
    OUTPUT_ROOT / 'phase3_setup_config.json',
]
setup_manifest = pd.DataFrame([
    {'File': path.name, 'Bytes': path.stat().st_size}
    for path in setup_files
])
setup_manifest.to_csv(OUTPUT_ROOT / 'phase3_setup_manifest.csv', index=False)
assert len(pd.read_csv(OUTPUT_ROOT / 'phase3_contract_validation.csv')) == 18
assert len(pd.read_csv(OUTPUT_ROOT / 'phase3_task_definitions.csv')) == 4
assert len(pd.read_csv(OUTPUT_ROOT / 'phase3_setup_manifest.csv')) == 3
display(setup_manifest)

,File,Bytes
0,phase3_contract_validation.csv,1084
1,phase3_task_definitions.csv,1028
2,phase3_setup_config.json,1022


## Step 6 — Popularity baselines

Two deterministic global baselines are compared for every dataset/task/relevance combination:

- `popularity_discovery_early`: discovery-early interaction counts from the frozen candidate catalogue.
- `popularity_discovery_future`: the number of discovery learners with the relevant future item. Validation-future labels are excluded before scores are fitted.

Recommendations are generated directly as sparse top-20 rows. For `novel_only`, the ranked catalogue is scanned until each learner has up to 20 unseen items; no dense learner-by-item matrix is constructed.

In [8]:
MAX_RECOMMENDATIONS = max(METRIC_KS)

def load_task_tables(task_definition):
    directory = DATASET_DIRS[task_definition['Dataset']]
    return {
        'learners': pd.read_parquet(directory / 'learner_splits.parquet'),
        'catalog': pd.read_parquet(directory / task_definition['CatalogFile']),
        'early_history': pd.read_parquet(directory / task_definition['EarlyHistoryFile']),
        'future_relevance': pd.read_parquet(directory / task_definition['FutureRelevanceFile']),
    }

def top_k_sparse(scored_candidates, max_k=MAX_RECOMMENDATIONS):
    required = {'learner_id', 'item_id', 'score'}
    if not required.issubset(scored_candidates.columns):
        raise ValueError(f'Scored candidates require columns: {sorted(required)}')
    if scored_candidates.empty:
        return pd.DataFrame(columns=['learner_id', 'item_id', 'score', 'rank'])
    assert not scored_candidates.duplicated(['learner_id', 'item_id']).any()
    ranked = scored_candidates[['learner_id', 'item_id', 'score']].sort_values(
        ['learner_id', 'score', 'item_id'],
        ascending=[True, False, True],
        kind='mergesort',
    )
    ranked = ranked.groupby('learner_id', sort=False).head(max_k).copy()
    ranked['rank'] = ranked.groupby('learner_id', sort=False).cumcount() + 1
    return ranked

def global_top_k_recommendations(
    validation_learners, item_scores, early_history, candidate_policy,
    max_k=MAX_RECOMMENDATIONS,
):
    if candidate_policy not in CANDIDATE_POLICIES:
        raise ValueError(f'Unknown candidate policy: {candidate_policy}')
    required = {'item_id', 'score'}
    if not required.issubset(item_scores.columns):
        raise ValueError('Item scores require item_id and score.')
    assert not item_scores['item_id'].duplicated().any()
    assert np.isfinite(pd.to_numeric(item_scores['score'], errors='coerce')).all()
    ranking = list(
        item_scores[['item_id', 'score']]
        .sort_values(['score', 'item_id'], ascending=[False, True], kind='mergesort')
        .itertuples(index=False, name=None)
    )
    learner_ids = sorted(pd.Series(validation_learners).astype(str).unique())
    seen_by_learner = {}
    if candidate_policy == 'novel_only':
        validation_set = set(learner_ids)
        seen = early_history[
            early_history['in_candidate_catalog']
            & early_history['learner_id'].astype(str).isin(validation_set)
        ][['learner_id', 'item_id']].drop_duplicates()
        seen_by_learner = seen.groupby('learner_id')['item_id'].agg(set).to_dict()

    rows = []
    for learner_id in learner_ids:
        excluded = seen_by_learner.get(learner_id, set())
        rank = 0
        for item_id, score in ranking:
            if item_id in excluded:
                continue
            rank += 1
            rows.append((learner_id, item_id, float(score), rank))
            if rank == max_k:
                break
    recommendations = pd.DataFrame(
        rows, columns=['learner_id', 'item_id', 'score', 'rank']
    )
    assert not recommendations.duplicated(['learner_id', 'item_id']).any()
    assert recommendations.groupby('learner_id').size().le(max_k).all()
    return recommendations

def discovery_early_popularity_scores(catalog):
    scores = catalog[
        ['item_id', 'training_interactions', 'training_learners', 'training_popularity']
    ].copy()
    scores['score'] = scores['training_interactions'].astype(float)
    scores['normalised_score'] = scores['training_popularity'].astype(float)
    scores['score_source'] = 'discovery_early_interactions'
    return scores

def discovery_future_popularity_scores(
    relevance, learners, catalog, relevance_column,
):
    discovery_ids = set(
        learners.loc[learners['cohort'].eq('discovery'), 'learner_id'].astype(str)
    )
    validation_ids = set(
        learners.loc[learners['cohort'].eq('validation'), 'learner_id'].astype(str)
    )
    training_labels = relevance[
        relevance['learner_id'].astype(str).isin(discovery_ids)
        & relevance['in_candidate_catalog']
    ][['learner_id', 'item_id', relevance_column]].copy()
    assert set(training_labels['learner_id']).issubset(discovery_ids)
    assert set(training_labels['learner_id']).isdisjoint(validation_ids)
    assert not training_labels.duplicated(['learner_id', 'item_id']).any()
    relevant_labels = training_labels[training_labels[relevance_column].eq(1)]
    counts = (
        relevant_labels.groupby('item_id')['learner_id']
        .nunique().rename('discovery_relevant_learners').reset_index()
    )
    scores = catalog[['item_id']].merge(counts, on='item_id', how='left')
    scores['discovery_relevant_learners'] = (
        scores['discovery_relevant_learners'].fillna(0).astype('int64')
    )
    scores['score'] = scores['discovery_relevant_learners'].astype(float)
    total = scores['score'].sum()
    scores['normalised_score'] = scores['score'] / total if total else 0.0
    scores['score_source'] = f'discovery_future_{relevance_column}'
    return scores

In [9]:
popularity_metric_parts = []
popularity_recommendation_parts = []
popularity_score_parts = []

for task_definition in TASK_DEFINITIONS:
    dataset = task_definition['Dataset']
    task = task_definition['Task']
    tables = load_task_tables(task_definition)
    learners = tables['learners']
    catalog = tables['catalog']
    early_history = tables['early_history']
    future_relevance = tables['future_relevance']
    validation_ids = learners.loc[
        learners['cohort'].eq('validation'), 'learner_id'
    ].astype(str)

    early_scores = discovery_early_popularity_scores(catalog)
    assert_candidate_subset(early_scores[['item_id', 'score']], catalog)

    for relevance_name, relevance_column in RELEVANCE_DEFINITIONS.items():
        future_scores = discovery_future_popularity_scores(
            future_relevance, learners, catalog, relevance_column
        )
        models = {
            'popularity_discovery_early': early_scores,
            'popularity_discovery_future': future_scores,
        }
        for model_name, item_scores in models.items():
            score_output = item_scores.copy()
            score_output.insert(0, 'RelevanceDefinition', relevance_name)
            score_output.insert(0, 'Model', model_name)
            score_output.insert(0, 'Task', task)
            score_output.insert(0, 'Dataset', dataset)
            popularity_score_parts.append(score_output)

            for candidate_policy in CANDIDATE_POLICIES:
                recommendation_started = time.perf_counter()
                recommendations = global_top_k_recommendations(
                    validation_ids, item_scores, early_history,
                    candidate_policy, MAX_RECOMMENDATIONS,
                )
                assert_candidate_subset(recommendations, catalog)
                recommendation_runtime = time.perf_counter() - recommendation_started
                metrics = evaluate_recommendations(
                    recommendations, future_relevance, catalog, learners,
                    relevance_column, task_definition['EvaluableColumn'],
                    task_definition['ColdStartColumn'], dataset, task,
                    candidate_policy, relevance_name,
                )
                metrics.insert(4, 'Model', model_name)
                metrics['RecommendationBuildRuntimeSeconds'] = recommendation_runtime
                popularity_metric_parts.append(metrics)

                recommendation_output = recommendations.copy()
                recommendation_output.insert(0, 'RelevanceDefinition', relevance_name)
                recommendation_output.insert(0, 'CandidatePolicy', candidate_policy)
                recommendation_output.insert(0, 'Model', model_name)
                recommendation_output.insert(0, 'Task', task)
                recommendation_output.insert(0, 'Dataset', dataset)
                popularity_recommendation_parts.append(recommendation_output)

popularity_metrics = pd.concat(popularity_metric_parts, ignore_index=True)
popularity_recommendations = pd.concat(
    popularity_recommendation_parts, ignore_index=True
)
popularity_scores = pd.concat(popularity_score_parts, ignore_index=True)
assert set(popularity_metrics['Model']) == {
    'popularity_discovery_early', 'popularity_discovery_future'
}
assert set(popularity_metrics['CandidatePolicy']) == set(CANDIDATE_POLICIES)
assert set(popularity_metrics['RelevanceDefinition']) == set(RELEVANCE_DEFINITIONS)
display(popularity_metrics.sort_values(
    ['Dataset', 'Task', 'RelevanceDefinition', 'CandidatePolicy', 'Model', 'Segment', 'K']
))

,Dataset,Task,CandidatePolicy,RelevanceDefinition,Model,Segment,K,PrecisionAtK,RecallAtK,NDCGAtK,MAPAtK,HitRateAtK,CatalogCoverageAtK,MeanRecommendations,EvaluatedLearners,EvaluableRate,EvaluationScope,RecommendationMemoryBytes,MetricRuntimeSeconds,RecommendationBuildRuntimeSeconds
0,ASSISTments,problem,all_supported,attempted,popularity_discovery_early,all,5,0.011341,0.003534,0.011477,0.009014,0.021014,0.000126,5.000000,5996,0.899355,validation,17866552,3.947271,0.290237
1,ASSISTments,problem,all_supported,attempted,popularity_discovery_early,all,10,0.011474,0.007234,0.011900,0.008486,0.027352,0.000251,10.000000,5996,0.899355,validation,17866552,3.947271,0.290237
2,ASSISTments,problem,all_supported,attempted,popularity_discovery_early,all,20,0.011116,0.013276,0.014485,0.009746,0.032188,0.000503,20.000000,5996,0.899355,validation,17866552,3.947271,0.290237
3,ASSISTments,problem,all_supported,attempted,popularity_discovery_early,cold_start,5,0.000000,0.000000,0.000000,0.000000,0.000000,0.000126,5.000000,21,0.899355,validation,17866552,3.947271,0.290237
4,ASSISTments,problem,all_supported,attempted,popularity_discovery_early,cold_start,10,0.000000,0.000000,0.000000,0.000000,0.000000,0.000251,10.000000,21,0.899355,validation,17866552,3.947271,0.290237
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
235,KDD,skill,novel_only,successful,popularity_discovery_future,all,10,0.294318,0.597980,0.509490,0.402189,0.818182,0.808081,9.965909,88,0.778761,validation,462925,0.184480,0.022086
236,KDD,skill,novel_only,successful,popularity_discovery_future,all,20,0.204545,0.783098,0.566299,0.434034,0.931818,0.929293,19.715909,88,0.778761,validation,462925,0.184480,0.022086
237,KDD,skill,novel_only,successful,popularity_discovery_future,non_cold_start,5,0.377273,0.382336,0.452428,0.384201,0.715909,0.585859,5.000000,88,0.778761,validation,462925,0.184480,0.022086
238,KDD,skill,novel_only,successful,popularity_discovery_future,non_cold_start,10,0.294318,0.597980,0.509490,0.402189,0.818182,0.808081,9.965909,88,0.778761,validation,462925,0.184480,0.022086


## Step 7 — Weak-skill baseline

This baseline applies only to skill recommendation. It scores validation learners' supported `weak` and `developing` skills as `(1 - empirical_bayes_mastery) * mastery_evidence_confidence`.

`insufficient_evidence` remains a separate diagnostic state and is never interpreted as demonstrated weakness. The model has no unobserved-skill transfer signal, so `novel_only` correctly produces empty recommendation lists instead of adding an undeclared popularity backfill.

In [10]:
WEAK_SKILL_STATES = {'weak', 'developing'}
weak_skill_metric_parts = []
weak_skill_recommendation_parts = []
weak_skill_score_parts = []
weak_skill_diagnostic_parts = []

for task_definition in [row for row in TASK_DEFINITIONS if row['Task'] == 'skill']:
    dataset = task_definition['Dataset']
    tables = load_task_tables(task_definition)
    learners = tables['learners']
    catalog = tables['catalog']
    early_history = tables['early_history']
    future_relevance = tables['future_relevance']
    validation_ids = set(
        learners.loc[learners['cohort'].eq('validation'), 'learner_id'].astype(str)
    )
    validation_history = early_history[
        early_history['learner_id'].astype(str).isin(validation_ids)
        & early_history['in_candidate_catalog']
    ].copy()

    diagnostics = (
        validation_history.groupby('skill_state', dropna=False)
        .size().rename('LearnerSkillRows').reset_index()
    )
    diagnostics.insert(0, 'Dataset', dataset)
    diagnostics['UsedAsWeaknessEvidence'] = diagnostics['skill_state'].isin(
        WEAK_SKILL_STATES
    )
    weak_skill_diagnostic_parts.append(diagnostics)

    scored = validation_history[
        validation_history['skill_state'].isin(WEAK_SKILL_STATES)
    ][
        ['learner_id', 'item_id', 'empirical_bayes_mastery',
         'mastery_evidence_confidence', 'skill_state']
    ].copy()
    assert not scored['skill_state'].eq('insufficient_evidence').any()
    assert scored['empirical_bayes_mastery'].between(0, 1).all()
    assert scored['mastery_evidence_confidence'].between(0, 1).all()
    scored['score'] = (
        (1.0 - scored['empirical_bayes_mastery'])
        * scored['mastery_evidence_confidence']
    )
    assert np.isfinite(scored['score']).all()
    assert_candidate_subset(scored, catalog)

    score_output = scored.copy()
    score_output.insert(0, 'Model', 'weak_skill_mastery_confidence')
    score_output.insert(0, 'Task', 'skill')
    score_output.insert(0, 'Dataset', dataset)
    weak_skill_score_parts.append(score_output)

    for candidate_policy in CANDIDATE_POLICIES:
        recommendation_started = time.perf_counter()
        policy_scores = apply_candidate_policy(
            scored, validation_history, candidate_policy
        )
        recommendations = top_k_sparse(policy_scores, MAX_RECOMMENDATIONS)
        if candidate_policy == 'novel_only':
            assert recommendations.empty
        recommendation_runtime = time.perf_counter() - recommendation_started
        assert_candidate_subset(recommendations, catalog)

        for relevance_name, relevance_column in RELEVANCE_DEFINITIONS.items():
            metrics = evaluate_recommendations(
                recommendations, future_relevance, catalog, learners,
                relevance_column, task_definition['EvaluableColumn'],
                task_definition['ColdStartColumn'], dataset, 'skill',
                candidate_policy, relevance_name,
            )
            metrics.insert(4, 'Model', 'weak_skill_mastery_confidence')
            metrics['RecommendationBuildRuntimeSeconds'] = recommendation_runtime
            metrics['PolicyInterpretation'] = np.where(
                metrics['CandidatePolicy'].eq('novel_only'),
                'no_unseen_transfer_signal', 'personalised_weakness_ranking',
            )
            weak_skill_metric_parts.append(metrics)

            recommendation_output = recommendations.copy()
            recommendation_output.insert(0, 'RelevanceDefinition', relevance_name)
            recommendation_output.insert(0, 'CandidatePolicy', candidate_policy)
            recommendation_output.insert(0, 'Model', 'weak_skill_mastery_confidence')
            recommendation_output.insert(0, 'Task', 'skill')
            recommendation_output.insert(0, 'Dataset', dataset)
            weak_skill_recommendation_parts.append(recommendation_output)

weak_skill_metrics = pd.concat(weak_skill_metric_parts, ignore_index=True)
weak_skill_recommendations = pd.concat(
    [part for part in weak_skill_recommendation_parts if not part.empty],
    ignore_index=True,
)
weak_skill_scores = pd.concat(weak_skill_score_parts, ignore_index=True)
weak_skill_diagnostics = pd.concat(
    weak_skill_diagnostic_parts, ignore_index=True
)
assert not weak_skill_diagnostics.loc[
    weak_skill_diagnostics['skill_state'].eq('insufficient_evidence'),
    'UsedAsWeaknessEvidence',
].any()
display(weak_skill_metrics.sort_values(
    ['Dataset', 'RelevanceDefinition', 'CandidatePolicy', 'Segment', 'K']
))
display(weak_skill_diagnostics.sort_values(['Dataset', 'skill_state']))

,Dataset,Task,CandidatePolicy,RelevanceDefinition,Model,Segment,K,PrecisionAtK,RecallAtK,NDCGAtK,...,HitRateAtK,CatalogCoverageAtK,MeanRecommendations,EvaluatedLearners,EvaluableRate,EvaluationScope,RecommendationMemoryBytes,MetricRuntimeSeconds,RecommendationBuildRuntimeSeconds,PolicyInterpretation
0,ASSISTments,skill,all_supported,attempted,weak_skill_mastery_confidence,all,5,0.244773,0.197409,0.321089,...,0.616012,0.844720,3.291688,3922,0.588271,validation,2980604,3.032519,0.036416,personalised_weakness_ranking
1,ASSISTments,skill,all_supported,attempted,weak_skill_mastery_confidence,all,10,0.168256,0.227048,0.295214,...,0.636410,0.869565,4.569862,3922,0.588271,validation,2980604,3.032519,0.036416,personalised_weakness_ranking
2,ASSISTments,skill,all_supported,attempted,weak_skill_mastery_confidence,all,20,0.097081,0.238813,0.275014,...,0.639215,0.875776,5.279704,3922,0.588271,validation,2980604,3.032519,0.036416,personalised_weakness_ranking
3,ASSISTments,skill,all_supported,attempted,weak_skill_mastery_confidence,cold_start,5,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,133,0.588271,validation,2980604,3.032519,0.036416,personalised_weakness_ranking
4,ASSISTments,skill,all_supported,attempted,weak_skill_mastery_confidence,cold_start,10,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,133,0.588271,validation,2980604,3.032519,0.036416,personalised_weakness_ranking
5,ASSISTments,skill,all_supported,attempted,weak_skill_mastery_confidence,cold_start,20,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,133,0.588271,validation,2980604,3.032519,0.036416,personalised_weakness_ranking
6,ASSISTments,skill,all_supported,attempted,weak_skill_mastery_confidence,non_cold_start,5,0.253365,0.204338,0.332360,...,0.637635,0.844720,3.407231,3789,0.588271,validation,2980604,3.032519,0.036416,personalised_weakness_ranking
7,ASSISTments,skill,all_supported,attempted,weak_skill_mastery_confidence,non_cold_start,10,0.174162,0.235017,0.305576,...,0.658749,0.869565,4.730272,3789,0.588271,validation,2980604,3.032519,0.036416,personalised_weakness_ranking
8,ASSISTments,skill,all_supported,attempted,weak_skill_mastery_confidence,non_cold_start,20,0.100488,0.247196,0.284667,...,0.661652,0.875776,5.465030,3789,0.588271,validation,2980604,3.032519,0.036416,personalised_weakness_ranking
18,ASSISTments,skill,novel_only,attempted,weak_skill_mastery_confidence,all,5,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,3441,0.516124,validation,132,1.659333,0.040559,no_unseen_transfer_signal


,Dataset,skill_state,LearnerSkillRows,UsedAsWeaknessEvidence
0,ASSISTments,developing,14898,True
1,ASSISTments,insufficient_evidence,17882,False
2,ASSISTments,mastered,12039,False
3,ASSISTments,weak,6963,True
4,KDD,developing,1001,True
5,KDD,insufficient_evidence,500,False
6,KDD,mastered,1175,False
7,KDD,weak,1094,True


## Save and verify Steps 6-7 artifacts

These are staged baseline artifacts. The consolidated Phase 3 filenames required by Step 15 will be produced only after all baseline families are implemented.

In [11]:
steps_6_7_metrics = pd.concat(
    [popularity_metrics, weak_skill_metrics], ignore_index=True, sort=False
)
steps_6_7_recommendations = pd.concat(
    [popularity_recommendations, weak_skill_recommendations],
    ignore_index=True, sort=False,
)

steps_6_7_paths = {
    'metrics': OUTPUT_ROOT / 'steps_6_7_metrics.csv',
    'recommendations': OUTPUT_ROOT / 'steps_6_7_recommendations.parquet',
    'popularity_scores': OUTPUT_ROOT / 'step6_popularity_scores.parquet',
    'weak_skill_scores': OUTPUT_ROOT / 'step7_weak_skill_scores.parquet',
    'weak_skill_diagnostics': OUTPUT_ROOT / 'step7_weak_skill_diagnostics.csv',
    'config': OUTPUT_ROOT / 'steps_6_7_config.json',
}
steps_6_7_metrics.to_csv(steps_6_7_paths['metrics'], index=False)
steps_6_7_recommendations.to_parquet(
    steps_6_7_paths['recommendations'], index=False, compression='snappy'
)
popularity_scores.to_parquet(
    steps_6_7_paths['popularity_scores'], index=False, compression='snappy'
)
weak_skill_scores.to_parquet(
    steps_6_7_paths['weak_skill_scores'], index=False, compression='snappy'
)
weak_skill_diagnostics.to_csv(
    steps_6_7_paths['weak_skill_diagnostics'], index=False
)

steps_6_7_config = {
    'implemented_steps': [6, 7],
    'metric_ks': list(METRIC_KS),
    'maximum_saved_recommendations_per_learner': MAX_RECOMMENDATIONS,
    'popularity_models': [
        'popularity_discovery_early', 'popularity_discovery_future'
    ],
    'discovery_future_training_learners_only': True,
    'validation_future_role': 'evaluation_only',
    'candidate_policies': list(CANDIDATE_POLICIES),
    'relevance_definitions': RELEVANCE_DEFINITIONS,
    'weak_skill_model': 'weak_skill_mastery_confidence',
    'weak_skill_formula': '(1 - empirical_bayes_mastery) * mastery_evidence_confidence',
    'weak_skill_states': sorted(WEAK_SKILL_STATES),
    'insufficient_evidence_handling': 'diagnostic_only_not_weakness',
    'weak_skill_novel_only_handling': 'empty_no_unseen_transfer_signal',
    'dense_learner_item_matrix_constructed': False,
}
with open(steps_6_7_paths['config'], 'w', encoding='utf-8') as file:
    json.dump(steps_6_7_config, file, indent=2)

steps_6_7_manifest = pd.DataFrame([
    {
        'Artifact': name,
        'File': path.name,
        'Rows': (
            pq.ParquetFile(path).metadata.num_rows
            if path.suffix == '.parquet'
            else (len(pd.read_csv(path)) if path.suffix == '.csv' else 1)
        ),
        'Bytes': path.stat().st_size,
    }
    for name, path in steps_6_7_paths.items()
])
steps_6_7_manifest.to_csv(
    OUTPUT_ROOT / 'steps_6_7_artifact_manifest.csv', index=False
)
assert len(steps_6_7_manifest) == 6
assert pq.ParquetFile(
    steps_6_7_paths['recommendations']
).metadata.num_rows == len(steps_6_7_recommendations)
assert len(pd.read_csv(steps_6_7_paths['metrics'])) == len(steps_6_7_metrics)
display(steps_6_7_manifest)

,Artifact,File,Rows,Bytes
0,metrics,steps_6_7_metrics.csv,300,82367
1,recommendations,steps_6_7_recommendations.parquet,2215406,1109216
2,popularity_scores,step6_popularity_scores.parquet,163576,1193533
3,weak_skill_scores,step7_weak_skill_scores.parquet,23956,339990
4,weak_skill_diagnostics,step7_weak_skill_diagnostics.csv,8,304
5,config,steps_6_7_config.json,1,909


## Step 8 — Content-based problem recommendation

This step defines reusable sparse learner-problem content components from early weak/developing skill evidence and early problem metadata. Candidate problems come only from the discovery-early problem catalogue and discovery-early problem-to-skill map.

The score combines weak-skill alignment, difficulty suitability, problem-type affinity and hierarchy affinity. Discovery-early popularity is used only as a deterministic numerical tie-breaker. Difficulty tolerance is selected on discovery learners in Step 13 before the only validation evaluation.

In [12]:
CONTENT_WEIGHTS = {
    'weak_skill_alignment': 0.65,
    'difficulty_suitability': 0.20,
    'problem_type_affinity': 0.10,
    'hierarchy_affinity': 0.05,
}
POPULARITY_TIE_BREAK_WEIGHT = 1e-9

def content_components_for_learners(task_definition, query_learner_ids):
    dataset = task_definition['Dataset']
    directory = DATASET_DIRS[dataset]
    skill_history = pd.read_parquet(directory / 'early_skill_history.parquet')
    problem_history = pd.read_parquet(directory / 'early_problem_history.parquet')
    problem_catalog = pd.read_parquet(directory / 'problem_catalog.parquet')
    problem_skill_map = pd.read_parquet(directory / 'problem_skill_map.parquet')

    assert problem_catalog['catalog_source'].eq('discovery_early_only').all()
    assert problem_skill_map['mapping_source'].eq('discovery_early_only').all()
    query_ids = set(pd.Series(query_learner_ids).astype(str))
    skill_need = skill_history[
        skill_history['learner_id'].astype(str).isin(query_ids)
        & skill_history['in_candidate_catalog']
        & skill_history['skill_state'].isin(WEAK_SKILL_STATES)
    ][
        ['learner_id', 'item_id', 'empirical_bayes_mastery',
         'mastery_evidence_confidence', 'skill_state']
    ].copy()
    skill_need['weakness'] = (
        (1.0 - skill_need['empirical_bayes_mastery'])
        * skill_need['mastery_evidence_confidence']
    )
    assert np.isfinite(skill_need['weakness']).all()

    mapped = skill_need.merge(
        problem_skill_map[
            ['problem_item_id', 'skill_item_id', 'association_share']
        ],
        left_on='item_id', right_on='skill_item_id', how='inner',
        validate='many_to_many',
    )
    mapped['alignment_contribution'] = (
        mapped['weakness'] * mapped['association_share']
    )
    mapped['mastery_contribution'] = (
        mapped['empirical_bayes_mastery'] * mapped['association_share']
    )
    candidate_scores = mapped.groupby(
        ['learner_id', 'problem_item_id'], sort=False
    ).agg(
        weak_skill_alignment=('alignment_contribution', 'sum'),
        weighted_mastery=('mastery_contribution', 'sum'),
        mapped_association_coverage=('association_share', 'sum'),
        matched_weak_skills=('skill_item_id', 'nunique'),
    ).reset_index().rename(columns={'problem_item_id': 'item_id'})
    candidate_scores['profile_mastery'] = (
        candidate_scores['weighted_mastery']
        / candidate_scores['mapped_association_coverage'].clip(lower=1e-12)
    ).clip(0, 1)

    catalog_metadata = problem_catalog[
        ['item_id', 'difficulty_proxy', 'problem_type', 'hierarchy',
         'training_popularity']
    ].copy()
    catalog_metadata['difficulty_proxy'] = (
        catalog_metadata['difficulty_proxy'].fillna(0.5).clip(0, 1)
    )
    catalog_metadata['problem_type'] = (
        catalog_metadata['problem_type'].fillna('__missing__').astype(str)
    )
    catalog_metadata['hierarchy'] = (
        catalog_metadata['hierarchy'].fillna('__missing__').astype(str)
    )

    early_problem_profiles = problem_history[
        problem_history['learner_id'].astype(str).isin(query_ids)
        & problem_history['in_candidate_catalog']
    ][['learner_id', 'item_id', 'early_interaction_count']].merge(
        catalog_metadata[['item_id', 'problem_type', 'hierarchy']],
        on='item_id', how='inner', validate='many_to_one',
    )
    profile_totals = early_problem_profiles.groupby('learner_id')[
        'early_interaction_count'
    ].transform('sum').clip(lower=1)
    early_problem_profiles['interaction_share'] = (
        early_problem_profiles['early_interaction_count'] / profile_totals
    )
    type_affinity = early_problem_profiles.groupby(
        ['learner_id', 'problem_type'], sort=False
    )['interaction_share'].sum().rename('problem_type_affinity').reset_index()
    hierarchy_affinity = early_problem_profiles.groupby(
        ['learner_id', 'hierarchy'], sort=False
    )['interaction_share'].sum().rename('hierarchy_affinity').reset_index()

    candidate_scores = candidate_scores.merge(
        catalog_metadata, on='item_id', how='inner', validate='many_to_one'
    ).merge(
        type_affinity, on=['learner_id', 'problem_type'], how='left',
        validate='many_to_one',
    ).merge(
        hierarchy_affinity, on=['learner_id', 'hierarchy'], how='left',
        validate='many_to_one',
    )
    candidate_scores[['problem_type_affinity', 'hierarchy_affinity']] = (
        candidate_scores[['problem_type_affinity', 'hierarchy_affinity']]
        .fillna(0.0)
    )
    assert_candidate_subset(candidate_scores, problem_catalog)
    return candidate_scores, problem_catalog

In [13]:
CONTENT_DIFFICULTY_TOLERANCES = (0.25, 0.50, 1.00)
assert np.isclose(sum(CONTENT_WEIGHTS.values()), 1.0)

## Step 9 — Learner-neighbour collaborative filtering

Cosine nearest neighbours are fitted on sparse discovery-early learner-item histories. Candidate counts 20, 50 and 100 are selected using an internal discovery-train/discovery-tuning split, attempted relevance, all-supported candidates and NDCG@10.

This step stops after neighbour-count selection. Step 13 compares cosine and uniform-positive weighting, refits the selected configuration on every discovery learner, and performs the only validation evaluation. There is no dense learner-item score matrix or popularity backfill.

In [14]:
from scipy.sparse import csr_matrix, diags
from sklearn.model_selection import train_test_split
from sklearn.neighbors import NearestNeighbors

NEIGHBOR_CANDIDATES = (20, 50, 100)
DISCOVERY_TUNING_FRACTION = 0.20
NEIGHBOR_QUERY_BATCH_SIZE = 512

def build_sparse_history_matrix(
    history, learner_order, item_order, value_column='early_interaction_count',
):
    learner_order = [str(value) for value in learner_order]
    item_order = [str(value) for value in item_order]
    learner_index = {value: index for index, value in enumerate(learner_order)}
    item_index = {value: index for index, value in enumerate(item_order)}
    subset = history[
        history['learner_id'].astype(str).isin(learner_index)
        & history['item_id'].astype(str).isin(item_index)
        & history['in_candidate_catalog']
    ][['learner_id', 'item_id', value_column]].copy()
    rows = subset['learner_id'].astype(str).map(learner_index).to_numpy()
    columns = subset['item_id'].astype(str).map(item_index).to_numpy()
    values = np.log1p(
        pd.to_numeric(subset[value_column], errors='coerce').fillna(0).to_numpy(float)
    )
    matrix = csr_matrix(
        (values, (rows, columns)),
        shape=(len(learner_order), len(item_order)),
        dtype=np.float32,
    )
    matrix.eliminate_zeros()
    return matrix

def build_discovery_target_matrix(
    relevance, training_learner_order, item_order, relevance_column,
):
    training_learner_order = [str(value) for value in training_learner_order]
    item_order = [str(value) for value in item_order]
    learner_index = {
        value: index for index, value in enumerate(training_learner_order)
    }
    item_index = {value: index for index, value in enumerate(item_order)}
    labels = relevance[
        relevance['learner_id'].astype(str).isin(learner_index)
        & relevance['item_id'].astype(str).isin(item_index)
        & relevance['in_candidate_catalog']
        & relevance[relevance_column].eq(1)
    ][['learner_id', 'item_id']].drop_duplicates()
    assert set(labels['learner_id'].astype(str)).issubset(learner_index)
    rows = labels['learner_id'].astype(str).map(learner_index).to_numpy()
    columns = labels['item_id'].astype(str).map(item_index).to_numpy()
    matrix = csr_matrix(
        (np.ones(len(labels), dtype=np.float32), (rows, columns)),
        shape=(len(training_learner_order), len(item_order)),
        dtype=np.float32,
    )
    return matrix

def query_sparse_neighbors(training_matrix, query_matrix, maximum_neighbors):
    if training_matrix.shape[0] == 0:
        raise ValueError('Cannot fit neighbours without discovery learners.')
    usable_neighbors = min(maximum_neighbors, training_matrix.shape[0])
    model = NearestNeighbors(
        n_neighbors=usable_neighbors, metric='cosine',
        algorithm='brute', n_jobs=-1,
    )
    model.fit(training_matrix)
    distances, indices = model.kneighbors(query_matrix)
    similarities = np.clip(1.0 - distances, 0.0, 1.0)
    return indices, similarities

def aggregate_neighbor_recommendations(
    query_learner_order, training_target_matrix, neighbor_indices,
    neighbor_similarities, item_order, query_history, candidate_policy,
    max_k=MAX_RECOMMENDATIONS, batch_size=NEIGHBOR_QUERY_BATCH_SIZE,
):
    if candidate_policy not in CANDIDATE_POLICIES:
        raise ValueError(f'Unknown candidate policy: {candidate_policy}')
    query_learner_order = [str(value) for value in query_learner_order]
    item_order = [str(value) for value in item_order]
    seen_by_learner = {}
    if candidate_policy == 'novel_only':
        query_set = set(query_learner_order)
        seen = query_history[
            query_history['learner_id'].astype(str).isin(query_set)
            & query_history['in_candidate_catalog']
        ][['learner_id', 'item_id']].drop_duplicates()
        seen_by_learner = seen.groupby('learner_id')['item_id'].agg(set).to_dict()

    rows = []
    training_count = training_target_matrix.shape[0]
    for batch_start in range(0, len(query_learner_order), batch_size):
        batch_end = min(batch_start + batch_size, len(query_learner_order))
        batch_indices = neighbor_indices[batch_start:batch_end]
        batch_similarities = neighbor_similarities[batch_start:batch_end]
        local_rows = np.repeat(
            np.arange(batch_end - batch_start), batch_indices.shape[1]
        )
        weights = csr_matrix(
            (batch_similarities.ravel(), (local_rows, batch_indices.ravel())),
            shape=(batch_end - batch_start, training_count),
            dtype=np.float32,
        )
        weights.eliminate_zeros()
        weight_totals = np.asarray(weights.sum(axis=1)).ravel()
        inverse_totals = np.divide(
            1.0, weight_totals,
            out=np.zeros_like(weight_totals), where=weight_totals > 0,
        )
        weighted_scores = diags(inverse_totals) @ weights @ training_target_matrix
        weighted_scores = weighted_scores.tocsr()
        weighted_scores.eliminate_zeros()

        for local_index, learner_id in enumerate(
            query_learner_order[batch_start:batch_end]
        ):
            start = weighted_scores.indptr[local_index]
            end = weighted_scores.indptr[local_index + 1]
            item_indices = weighted_scores.indices[start:end]
            values = weighted_scores.data[start:end]
            excluded = seen_by_learner.get(learner_id, set())
            candidates = [
                (float(value), item_order[item_index])
                for item_index, value in zip(item_indices, values)
                if value > 0 and item_order[item_index] not in excluded
            ]
            candidates.sort(key=lambda value: (-value[0], value[1]))
            for rank, (score, item_id) in enumerate(
                candidates[:max_k], start=1
            ):
                rows.append((learner_id, item_id, score, rank))
    recommendations = pd.DataFrame(
        rows, columns=['learner_id', 'item_id', 'score', 'rank']
    )
    assert not recommendations.duplicated(['learner_id', 'item_id']).any()
    return recommendations

In [15]:
neighbor_tuning_parts = []
neighbor_selection_rows = []

for task_definition in TASK_DEFINITIONS:
    dataset = task_definition['Dataset']
    task = task_definition['Task']
    tables = load_task_tables(task_definition)
    learners = tables['learners']
    catalog = tables['catalog']
    early_history = tables['early_history']
    future_relevance = tables['future_relevance']
    item_order = sorted(catalog['item_id'].astype(str).unique())
    discovery_ids = sorted(
        learners.loc[learners['cohort'].eq('discovery'), 'learner_id'].astype(str)
    )
    validation_ids = sorted(
        learners.loc[learners['cohort'].eq('validation'), 'learner_id'].astype(str)
    )
    discovery_train_ids, discovery_tuning_ids = train_test_split(
        discovery_ids, test_size=DISCOVERY_TUNING_FRACTION,
        random_state=RANDOM_STATE,
    )
    discovery_train_ids = sorted(discovery_train_ids)
    discovery_tuning_ids = sorted(discovery_tuning_ids)
    assert set(discovery_train_ids).isdisjoint(discovery_tuning_ids)
    assert set(discovery_train_ids).isdisjoint(validation_ids)
    assert set(discovery_tuning_ids).isdisjoint(validation_ids)
    discovery_tuning_relevance = future_relevance[
        future_relevance['learner_id'].astype(str).isin(discovery_tuning_ids)
    ].copy()
    assert set(discovery_tuning_relevance['learner_id'].astype(str)).isdisjoint(
        validation_ids
    )

    tuning_training_matrix = build_sparse_history_matrix(
        early_history, discovery_train_ids, item_order
    )
    tuning_query_matrix = build_sparse_history_matrix(
        early_history, discovery_tuning_ids, item_order
    )
    tuning_neighbor_indices, tuning_neighbor_similarities = query_sparse_neighbors(
        tuning_training_matrix, tuning_query_matrix, max(NEIGHBOR_CANDIDATES)
    )
    tuning_target_matrix = build_discovery_target_matrix(
        future_relevance, discovery_train_ids, item_order,
        RELEVANCE_DEFINITIONS['attempted'],
    )
    task_tuning_parts = []
    for neighbor_count in NEIGHBOR_CANDIDATES:
        usable_neighbor_count = min(
            neighbor_count, tuning_neighbor_indices.shape[1]
        )
        tuning_recommendations = aggregate_neighbor_recommendations(
            discovery_tuning_ids, tuning_target_matrix,
            tuning_neighbor_indices[:, :usable_neighbor_count],
            tuning_neighbor_similarities[:, :usable_neighbor_count],
            item_order, early_history, 'all_supported',
        )
        tuning_metrics = evaluate_recommendations(
            tuning_recommendations, discovery_tuning_relevance, catalog, learners,
            RELEVANCE_DEFINITIONS['attempted'],
            task_definition['EvaluableColumn'], task_definition['ColdStartColumn'],
            dataset, task, 'all_supported', 'attempted', ks=(10,),
            evaluation_learner_ids=discovery_tuning_ids,
        )
        tuning_metrics.insert(4, 'Model', 'learner_neighbor_cf_tuning')
        tuning_metrics['NeighborCount'] = neighbor_count
        tuning_metrics['ActualNeighborCount'] = usable_neighbor_count
        tuning_metrics['TuningSource'] = 'discovery_only'
        task_tuning_parts.append(tuning_metrics)
        neighbor_tuning_parts.append(tuning_metrics)

    task_tuning = pd.concat(task_tuning_parts, ignore_index=True)
    selection_candidates = task_tuning[
        task_tuning['Segment'].eq('all') & task_tuning['K'].eq(10)
    ].sort_values(
        ['NDCGAtK', 'NeighborCount'], ascending=[False, True],
        kind='mergesort',
    )
    selected_neighbors = int(selection_candidates.iloc[0]['NeighborCount'])
    neighbor_selection_rows.append({
        'Dataset': dataset,
        'Task': task,
        'SelectedNeighbors': selected_neighbors,
        'SelectionMetric': 'NDCGAtK',
        'SelectionK': 10,
        'SelectionRelevance': 'attempted',
        'SelectionCandidatePolicy': 'all_supported',
        'DiscoveryTrainingLearners': len(discovery_train_ids),
        'DiscoveryTuningLearners': len(discovery_tuning_ids),
        'ValidationLearnersUntouched': len(validation_ids),
    })

neighbor_tuning_metrics = pd.concat(neighbor_tuning_parts, ignore_index=True)
neighbor_selection = pd.DataFrame(neighbor_selection_rows)
assert neighbor_selection[['Dataset', 'Task']].drop_duplicates().shape[0] == 4
display(neighbor_selection)
display(neighbor_tuning_metrics.sort_values(
    ['Dataset', 'Task', 'NeighborCount', 'Segment']
))

,Dataset,Task,SelectedNeighbors,SelectionMetric,SelectionK,SelectionRelevance,SelectionCandidatePolicy,DiscoveryTrainingLearners,DiscoveryTuningLearners,ValidationLearnersUntouched
0,ASSISTments,problem,20,NDCGAtK,10,attempted,all_supported,21334,5334,6667
1,ASSISTments,skill,20,NDCGAtK,10,attempted,all_supported,21334,5334,6667
2,KDD,problem,20,NDCGAtK,10,attempted,all_supported,361,91,113
3,KDD,skill,20,NDCGAtK,10,attempted,all_supported,361,91,113


,Dataset,Task,CandidatePolicy,RelevanceDefinition,Model,Segment,K,PrecisionAtK,RecallAtK,NDCGAtK,...,CatalogCoverageAtK,MeanRecommendations,EvaluatedLearners,EvaluableRate,EvaluationScope,RecommendationMemoryBytes,MetricRuntimeSeconds,NeighborCount,ActualNeighborCount,TuningSource
0,ASSISTments,problem,all_supported,attempted,learner_neighbor_cf_tuning,all,10,0.440946,0.265667,0.498437,...,0.194474,9.802667,4799,0.899700,provided_learner_ids,13023370,1.607299,20,20,discovery_only
1,ASSISTments,problem,all_supported,attempted,learner_neighbor_cf_tuning,cold_start,10,0.000000,0.000000,0.000000,...,0.000000,0.000000,13,0.899700,provided_learner_ids,13023370,1.607299,20,20,discovery_only
2,ASSISTments,problem,all_supported,attempted,learner_neighbor_cf_tuning,non_cold_start,10,0.442144,0.266388,0.499790,...,0.194474,9.829294,4786,0.899700,provided_learner_ids,13023370,1.607299,20,20,discovery_only
3,ASSISTments,problem,all_supported,attempted,learner_neighbor_cf_tuning,all,10,0.421671,0.252298,0.475364,...,0.145001,9.864972,4799,0.899700,provided_learner_ids,13369101,1.332136,50,50,discovery_only
4,ASSISTments,problem,all_supported,attempted,learner_neighbor_cf_tuning,cold_start,10,0.000000,0.000000,0.000000,...,0.000000,0.000000,13,0.899700,provided_learner_ids,13369101,1.332136,50,50,discovery_only
5,ASSISTments,problem,all_supported,attempted,learner_neighbor_cf_tuning,non_cold_start,10,0.422817,0.252983,0.476655,...,0.145001,9.891768,4786,0.899700,provided_learner_ids,13369101,1.332136,50,50,discovery_only
6,ASSISTments,problem,all_supported,attempted,learner_neighbor_cf_tuning,all,10,0.403522,0.239419,0.454653,...,0.116544,9.930402,4799,0.899700,provided_learner_ids,13540074,1.257910,100,100,discovery_only
7,ASSISTments,problem,all_supported,attempted,learner_neighbor_cf_tuning,cold_start,10,0.000000,0.000000,0.000000,...,0.000000,0.000000,13,0.899700,provided_learner_ids,13540074,1.257910,100,100,discovery_only
8,ASSISTments,problem,all_supported,attempted,learner_neighbor_cf_tuning,non_cold_start,10,0.404618,0.240070,0.455888,...,0.116544,9.957376,4786,0.899700,provided_learner_ids,13540074,1.257910,100,100,discovery_only
9,ASSISTments,skill,all_supported,attempted,learner_neighbor_cf_tuning,all,10,0.460660,0.685504,0.737441,...,0.962733,9.021161,3119,0.584739,provided_learner_ids,6720201,0.687335,20,20,discovery_only


## Step 10 — Cluster-based collaborative-filtering ablations

The selected Step 9 neighbour count is reused for two cluster-aware comparisons:

- `same_cluster_neighbor_cf`: neighbours are discovery learners from the query learner's cluster only.
- `cluster_popularity`: items are ranked by discovery-future relevance within each cluster.

ASSISTments uses its accepted clustering solution for comparison. Every KDD cluster result is labelled `exploratory_ablation_only` and cannot determine the production method.

In [16]:
def build_cluster_neighbor_cache(
    learners, early_history, catalog, selected_neighbors,
):
    item_order = sorted(catalog['item_id'].astype(str).unique())
    discovery = learners[learners['cohort'].eq('discovery')].copy()
    validation = learners[learners['cohort'].eq('validation')].copy()
    cache = []
    for cluster_value in sorted(validation['cluster'].unique(), key=str):
        training_ids = sorted(
            discovery.loc[discovery['cluster'].eq(cluster_value), 'learner_id']
            .astype(str)
        )
        query_ids = sorted(
            validation.loc[validation['cluster'].eq(cluster_value), 'learner_id']
            .astype(str)
        )
        if not query_ids or not training_ids:
            continue
        assert set(training_ids).isdisjoint(query_ids)
        training_matrix = build_sparse_history_matrix(
            early_history, training_ids, item_order
        )
        query_matrix = build_sparse_history_matrix(
            early_history, query_ids, item_order
        )
        indices, similarities = query_sparse_neighbors(
            training_matrix, query_matrix, selected_neighbors
        )
        cache.append({
            'cluster': cluster_value,
            'training_ids': training_ids,
            'query_ids': query_ids,
            'neighbor_indices': indices,
            'neighbor_similarities': similarities,
        })
    return cache, item_order

def same_cluster_recommendations(
    cache, item_order, relevance, early_history, relevance_column,
    candidate_policy,
):
    parts = []
    for entry in cache:
        target_matrix = build_discovery_target_matrix(
            relevance, entry['training_ids'], item_order, relevance_column
        )
        part = aggregate_neighbor_recommendations(
            entry['query_ids'], target_matrix, entry['neighbor_indices'],
            entry['neighbor_similarities'], item_order, early_history,
            candidate_policy,
        )
        parts.append(part)
    nonempty = [part for part in parts if not part.empty]
    if not nonempty:
        return pd.DataFrame(columns=['learner_id', 'item_id', 'score', 'rank'])
    recommendations = pd.concat(nonempty, ignore_index=True)
    assert not recommendations.duplicated(['learner_id', 'item_id']).any()
    return recommendations

def discovery_cluster_popularity_scores(
    relevance, learners, catalog, relevance_column,
):
    discovery = learners[learners['cohort'].eq('discovery')][
        ['learner_id', 'cluster']
    ].copy()
    validation_ids = set(
        learners.loc[learners['cohort'].eq('validation'), 'learner_id'].astype(str)
    )
    labels = relevance[
        relevance['learner_id'].astype(str).isin(
            set(discovery['learner_id'].astype(str))
        )
        & relevance['in_candidate_catalog']
        & relevance[relevance_column].eq(1)
    ][['learner_id', 'item_id']].drop_duplicates().merge(
        discovery, on='learner_id', how='inner', validate='many_to_one'
    )
    assert set(labels['learner_id'].astype(str)).isdisjoint(validation_ids)
    counts = labels.groupby(['cluster', 'item_id'])['learner_id'].nunique().rename(
        'discovery_cluster_relevant_learners'
    ).reset_index()
    clusters = sorted(learners['cluster'].unique(), key=str)
    complete_index = pd.MultiIndex.from_product(
        [clusters, sorted(catalog['item_id'].astype(str).unique())],
        names=['cluster', 'item_id'],
    ).to_frame(index=False)
    scores = complete_index.merge(
        counts, on=['cluster', 'item_id'], how='left', validate='one_to_one'
    )
    scores['discovery_cluster_relevant_learners'] = (
        scores['discovery_cluster_relevant_learners'].fillna(0).astype('int64')
    )
    scores['score'] = scores['discovery_cluster_relevant_learners'].astype(float)
    cluster_totals = scores.groupby('cluster')['score'].transform('sum')
    scores['normalised_score'] = 0.0
    positive_total = cluster_totals.gt(0)
    scores.loc[positive_total, 'normalised_score'] = (
        scores.loc[positive_total, 'score']
        / cluster_totals.loc[positive_total]
    )
    scores['score_source'] = f'discovery_cluster_future_{relevance_column}'
    return scores

def cluster_popularity_recommendations(
    learners, cluster_scores, early_history, candidate_policy,
):
    validation = learners[learners['cohort'].eq('validation')]
    parts = []
    for cluster_value, cluster_learners in validation.groupby('cluster'):
        item_scores = cluster_scores[
            cluster_scores['cluster'].eq(cluster_value)
        ][['item_id', 'score']]
        part = global_top_k_recommendations(
            cluster_learners['learner_id'], item_scores, early_history,
            candidate_policy,
        )
        parts.append(part)
    return pd.concat(parts, ignore_index=True) if parts else pd.DataFrame(
        columns=['learner_id', 'item_id', 'score', 'rank']
    )

In [17]:
cluster_metric_parts = []
cluster_recommendation_parts = []
cluster_popularity_score_parts = []
cluster_diagnostic_rows = []

for task_definition in TASK_DEFINITIONS:
    dataset = task_definition['Dataset']
    task = task_definition['Task']
    tables = load_task_tables(task_definition)
    learners = tables['learners']
    catalog = tables['catalog']
    early_history = tables['early_history']
    future_relevance = tables['future_relevance']
    selected_neighbors = int(neighbor_selection.loc[
        neighbor_selection['Dataset'].eq(dataset)
        & neighbor_selection['Task'].eq(task),
        'SelectedNeighbors',
    ].iloc[0])
    evidence_use = (
        'accepted_ablation' if dataset == 'ASSISTments'
        else 'exploratory_ablation_only'
    )

    cache_started = time.perf_counter()
    cluster_cache, item_order = build_cluster_neighbor_cache(
        learners, early_history, catalog, selected_neighbors
    )
    cache_runtime = time.perf_counter() - cache_started
    covered_validation_ids = set().union(*(
        set(entry['query_ids']) for entry in cluster_cache
    )) if cluster_cache else set()
    expected_validation_ids = set(
        learners.loc[learners['cohort'].eq('validation'), 'learner_id'].astype(str)
    )
    cluster_diagnostic_rows.append({
        'Dataset': dataset,
        'Task': task,
        'ClusterEvidenceUse': evidence_use,
        'ClustersWithTrainingAndValidationLearners': len(cluster_cache),
        'ValidationLearnersCoveredByClusterNeighbors': len(covered_validation_ids),
        'ValidationLearners': len(expected_validation_ids),
        'ClusterNeighborCacheRuntimeSeconds': cache_runtime,
    })

    for relevance_name, relevance_column in RELEVANCE_DEFINITIONS.items():
        cluster_popularity_scores = discovery_cluster_popularity_scores(
            future_relevance, learners, catalog, relevance_column
        )
        score_output = cluster_popularity_scores.copy()
        score_output.insert(0, 'RelevanceDefinition', relevance_name)
        score_output.insert(0, 'Model', 'cluster_popularity')
        score_output.insert(0, 'Task', task)
        score_output.insert(0, 'Dataset', dataset)
        score_output['ClusterEvidenceUse'] = evidence_use
        cluster_popularity_score_parts.append(score_output)

        for candidate_policy in CANDIDATE_POLICIES:
            same_cluster_started = time.perf_counter()
            same_cluster = same_cluster_recommendations(
                cluster_cache, item_order, future_relevance, early_history,
                relevance_column, candidate_policy,
            )
            same_cluster_runtime = time.perf_counter() - same_cluster_started
            cluster_popularity_started = time.perf_counter()
            cluster_popularity = cluster_popularity_recommendations(
                learners, cluster_popularity_scores, early_history,
                candidate_policy,
            )
            cluster_popularity_runtime = (
                time.perf_counter() - cluster_popularity_started
            )
            models = {
                'same_cluster_neighbor_cf': (
                    same_cluster, same_cluster_runtime
                ),
                'cluster_popularity': (
                    cluster_popularity, cluster_popularity_runtime
                ),
            }
            for model_name, (recommendations, recommendation_runtime) in models.items():
                assert_candidate_subset(recommendations, catalog)
                metrics = evaluate_recommendations(
                    recommendations, future_relevance, catalog, learners,
                    relevance_column, task_definition['EvaluableColumn'],
                    task_definition['ColdStartColumn'], dataset, task,
                    candidate_policy, relevance_name,
                )
                metrics.insert(4, 'Model', model_name)
                metrics['NeighborCount'] = (
                    selected_neighbors
                    if model_name == 'same_cluster_neighbor_cf' else np.nan
                )
                metrics['ClusterEvidenceUse'] = evidence_use
                metrics['RecommendationBuildRuntimeSeconds'] = recommendation_runtime
                cluster_metric_parts.append(metrics)

                output = recommendations.copy()
                output.insert(0, 'RelevanceDefinition', relevance_name)
                output.insert(0, 'CandidatePolicy', candidate_policy)
                output.insert(0, 'Model', model_name)
                output.insert(0, 'Task', task)
                output.insert(0, 'Dataset', dataset)
                output['ClusterEvidenceUse'] = evidence_use
                cluster_recommendation_parts.append(output)

cluster_metrics = pd.concat(cluster_metric_parts, ignore_index=True)
cluster_recommendations = pd.concat(
    cluster_recommendation_parts, ignore_index=True
)
cluster_popularity_scores = pd.concat(
    cluster_popularity_score_parts, ignore_index=True
)
cluster_diagnostics = pd.DataFrame(cluster_diagnostic_rows)
assert cluster_metrics.loc[
    cluster_metrics['Dataset'].eq('KDD'), 'ClusterEvidenceUse'
].eq('exploratory_ablation_only').all()
display(cluster_diagnostics)
display(cluster_metrics.sort_values(
    ['Dataset', 'Task', 'RelevanceDefinition', 'CandidatePolicy',
     'Segment', 'K', 'Model']
))

,Dataset,Task,ClusterEvidenceUse,ClustersWithTrainingAndValidationLearners,ValidationLearnersCoveredByClusterNeighbors,ValidationLearners,ClusterNeighborCacheRuntimeSeconds
0,ASSISTments,problem,accepted_ablation,2,6667,6667,7.849435
1,ASSISTments,skill,accepted_ablation,2,6667,6667,3.762147
2,KDD,problem,exploratory_ablation_only,3,113,113,0.124479
3,KDD,skill,exploratory_ablation_only,3,113,113,0.090706


,Dataset,Task,CandidatePolicy,RelevanceDefinition,Model,Segment,K,PrecisionAtK,RecallAtK,NDCGAtK,...,CatalogCoverageAtK,MeanRecommendations,EvaluatedLearners,EvaluableRate,EvaluationScope,RecommendationMemoryBytes,MetricRuntimeSeconds,NeighborCount,ClusterEvidenceUse,RecommendationBuildRuntimeSeconds
9,ASSISTments,problem,all_supported,attempted,cluster_popularity,all,5,0.031621,0.009281,0.031650,...,0.000251,5.000000,5996,0.899355,validation,17866552,4.948571,NaN,accepted_ablation,0.267035
0,ASSISTments,problem,all_supported,attempted,same_cluster_neighbor_cf,all,5,0.472515,0.167946,0.500354,...,0.147414,4.946631,5996,0.899355,validation,16111002,4.574769,20.0,accepted_ablation,3.184719
10,ASSISTments,problem,all_supported,attempted,cluster_popularity,all,10,0.031004,0.017859,0.031371,...,0.000503,10.000000,5996,0.899355,validation,17866552,4.948571,NaN,accepted_ablation,0.267035
1,ASSISTments,problem,all_supported,attempted,same_cluster_neighbor_cf,all,10,0.430871,0.263530,0.488105,...,0.227256,9.793696,5996,0.899355,validation,16111002,4.574769,20.0,accepted_ablation,3.184719
11,ASSISTments,problem,all_supported,attempted,cluster_popularity,all,20,0.031388,0.022721,0.037076,...,0.000855,20.000000,5996,0.899355,validation,17866552,4.948571,NaN,accepted_ablation,0.267035
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
231,KDD,skill,novel_only,successful,same_cluster_neighbor_cf,non_cold_start,5,0.475000,0.524995,0.618087,...,0.747475,4.965909,88,0.778761,validation,408671,0.099143,20.0,exploratory_ablation_only,0.079499
238,KDD,skill,novel_only,successful,cluster_popularity,non_cold_start,10,0.295455,0.595205,0.516252,...,0.818182,9.965909,88,0.778761,validation,462737,0.094505,NaN,exploratory_ablation_only,0.046832
232,KDD,skill,novel_only,successful,same_cluster_neighbor_cf,non_cold_start,10,0.361364,0.681684,0.657783,...,0.898990,9.886364,88,0.778761,validation,408671,0.099143,20.0,exploratory_ablation_only,0.079499
239,KDD,skill,novel_only,successful,cluster_popularity,non_cold_start,20,0.209091,0.773370,0.573659,...,0.969697,19.715909,88,0.778761,validation,462737,0.094505,NaN,exploratory_ablation_only,0.046832


## Save and verify Steps 8-10 selection and ablation artifacts

Content evaluation is deferred to Step 13.

In [18]:
steps_8_10_metrics = cluster_metrics.copy()
steps_8_10_recommendations = cluster_recommendations.copy()

steps_8_10_paths = {
    'metrics': OUTPUT_ROOT / 'steps_8_10_metrics.csv',
    'recommendations': OUTPUT_ROOT / 'steps_8_10_recommendations.parquet',
    'neighbor_tuning': OUTPUT_ROOT / 'step9_neighbor_tuning.csv',
    'neighbor_selection': OUTPUT_ROOT / 'step9_neighbor_selection.csv',
    'cluster_popularity_scores': OUTPUT_ROOT / 'step10_cluster_popularity_scores.parquet',
    'cluster_diagnostics': OUTPUT_ROOT / 'step10_cluster_diagnostics.csv',
    'config': OUTPUT_ROOT / 'steps_8_10_config.json',
}
steps_8_10_metrics.to_csv(steps_8_10_paths['metrics'], index=False)
steps_8_10_recommendations.to_parquet(
    steps_8_10_paths['recommendations'], index=False, compression='snappy'
)
neighbor_tuning_metrics.to_csv(
    steps_8_10_paths['neighbor_tuning'], index=False
)
neighbor_selection.to_csv(
    steps_8_10_paths['neighbor_selection'], index=False
)
cluster_popularity_scores.to_parquet(
    steps_8_10_paths['cluster_popularity_scores'],
    index=False, compression='snappy',
)
cluster_diagnostics.to_csv(
    steps_8_10_paths['cluster_diagnostics'], index=False
)

steps_8_10_config = {
    'implemented_steps': [8, 9, 10],
    'content_weights': CONTENT_WEIGHTS,
    'content_popularity_tie_break_weight': POPULARITY_TIE_BREAK_WEIGHT,
    'content_evaluation_deferred_until_discovery_selection': True,
    'content_catalog_and_mapping_source': 'discovery_early_only',
    'neighbor_candidates': list(NEIGHBOR_CANDIDATES),
    'neighbor_selection_metric': 'attempted_all_supported_NDCGAt10',
    'neighbor_tuning_source': 'discovery_only',
    'neighbor_selection': neighbor_selection.to_dict(orient='records'),
    'neighbor_count_tuning_similarity': 'cosine_on_log1p_early_interaction_counts',
    'unrestricted_cf_validation_deferred_until_weighting_selection': True,
    'neighbor_target_source': 'discovery_future_relevance_only',
    'cluster_models': ['same_cluster_neighbor_cf', 'cluster_popularity'],
    'cluster_role': 'ablation_only',
    'kdd_cluster_role': 'exploratory_ablation_only',
    'candidate_policies': list(CANDIDATE_POLICIES),
    'relevance_definitions': RELEVANCE_DEFINITIONS,
    'maximum_saved_recommendations_per_learner': MAX_RECOMMENDATIONS,
    'dense_learner_item_matrix_constructed': False,
    'validation_future_role': 'evaluation_only',
}
with open(steps_8_10_paths['config'], 'w', encoding='utf-8') as file:
    json.dump(steps_8_10_config, file, indent=2)

steps_8_10_manifest = pd.DataFrame([
    {
        'Artifact': name,
        'File': path.name,
        'Rows': (
            pq.ParquetFile(path).metadata.num_rows
            if path.suffix == '.parquet'
            else (len(pd.read_csv(path)) if path.suffix == '.csv' else 1)
        ),
        'Bytes': path.stat().st_size,
    }
    for name, path in steps_8_10_paths.items()
])
steps_8_10_manifest.to_csv(
    OUTPUT_ROOT / 'steps_8_10_artifact_manifest.csv', index=False
)
assert len(steps_8_10_manifest) == 7
assert pq.ParquetFile(
    steps_8_10_paths['recommendations']
).metadata.num_rows == len(steps_8_10_recommendations)
assert len(pd.read_csv(steps_8_10_paths['metrics'])) == len(steps_8_10_metrics)
assert len(pd.read_csv(steps_8_10_paths['neighbor_selection'])) == 4
display(steps_8_10_manifest)

,Artifact,File,Rows,Bytes
0,metrics,steps_8_10_metrics.csv,240,69682
1,recommendations,steps_8_10_recommendations.parquet,1806183,4998368
2,neighbor_tuning,step9_neighbor_tuning.csv,30,8619
3,neighbor_selection,step9_neighbor_selection.csv,4,446
4,cluster_popularity_scores,step10_cluster_popularity_scores.parquet,165484,948671
5,cluster_diagnostics,step10_cluster_diagnostics.csv,4,439
6,config,steps_8_10_config.json,1,2784


## Step 11 — Sequential transition artifacts

Releases the largest already-saved Step 6-10 tables, checks the two frozen raw-data paths and reloads only the raw columns required for ordering and item identity. It repeats the Phase 2 cleaning and chronological cutoff rules, assigns cohorts from the saved learner split and verifies each learner's reconstructed early-event count exactly. ASSISTments uses native skill IDs first, then the frozen discovery-early name map and the same text fallback so that it never relearns mappings.

In [ ]:
import gc

# Steps 6-10 are already persisted. Release their largest in-memory tables
# before reloading raw event data for sequence reconstruction.
for large_name in [
    'steps_6_7_recommendations', 'popularity_recommendations',
    'weak_skill_recommendations', 'popularity_scores',
    'steps_8_10_recommendations', 'cluster_recommendations',
    'cluster_popularity_scores',
]:
    globals().pop(large_name, None)
gc.collect()

ASSIST_RAW_PATH = Path(
    '/content/drive/MyDrive/datasets/2012-2013-data-with-predictions-4-final.csv'
)
KDD_RAW_PATH = Path(
    '/content/drive/MyDrive/datasets/algebra_2005_2006_train.txt'
)
EARLY_FRACTION = float(benchmark_config['early_fraction'])
RECENT_SOURCE_ITEMS = 3
for raw_path in [ASSIST_RAW_PATH, KDD_RAW_PATH]:
    if not raw_path.exists():
        raise FileNotFoundError(f'Missing raw sequence source: {raw_path}')

def normalise_sequence_learner_ids(values):
    return (
        values.astype(str).str.strip()
        .str.replace(r'^(-?\d+)\.0$', r'\1', regex=True)
    )

def normalise_sequence_skill_names(values):
    return (
        values.fillna('').astype(str).str.strip().str.casefold()
        .str.replace(r'\s+', ' ', regex=True)
    )

def load_assist_ordered_events():
    columns = [
        'problem_log_id', 'user_id', 'problem_id', 'skill', 'skill_id',
        'start_time', 'end_time', 'correct',
    ]
    raw = pd.read_csv(
        ASSIST_RAW_PATH, usecols=columns, low_memory=False
    )
    for column in [
        'problem_log_id', 'user_id', 'problem_id', 'skill_id', 'correct'
    ]:
        raw[column] = pd.to_numeric(raw[column], errors='coerce')
    raw['start_time'] = pd.to_datetime(raw['start_time'], errors='coerce')
    raw['end_time'] = pd.to_datetime(raw['end_time'], errors='coerce')
    raw['event_time'] = raw['start_time'].fillna(raw['end_time'])
    duplicate = raw['problem_log_id'].notna() & raw.duplicated(
        'problem_log_id', keep='first'
    )
    raw = raw.loc[~duplicate]
    raw = raw[
        raw['user_id'].notna() & raw['correct'].isin([0, 1])
    ].copy()
    raw['learner_id'] = normalise_sequence_learner_ids(raw['user_id'])
    raw['event_id'] = raw['problem_log_id']
    raw['problem_item'] = (
        'problem:' + raw['problem_id'].astype('Int64').astype(str)
    )
    raw.loc[raw['problem_id'].isna(), 'problem_item'] = pd.NA

    mapping = pd.read_parquet(
        DATASET_DIRS['ASSISTments'] / 'skill_name_id_map.parquet'
    )
    assert mapping['mapping_source'].eq('discovery_early_only').all()
    mapping_lookup = mapping.set_index('normalised_skill_name')[
        'mapped_skill_id'
    ]
    normalised_name = normalise_sequence_skill_names(raw['skill'])
    native_id = pd.to_numeric(raw['skill_id'], errors='coerce').astype('Float64')
    mapped_id = normalised_name.map(mapping_lookup).astype('Float64')
    resolved_id = native_id.fillna(mapped_id)
    invalid_names = {'', 'unknown', 'nan', 'none'}
    text_fallback = resolved_id.isna() & ~normalised_name.isin(invalid_names)
    skill_item = pd.Series(pd.NA, index=raw.index, dtype='object')
    skill_item.loc[resolved_id.notna()] = (
        'skill:id:'
        + resolved_id.loc[resolved_id.notna()].astype('Int64').astype(str)
    )
    skill_item.loc[text_fallback] = (
        'skill:text:' + normalised_name.loc[text_fallback]
    )
    raw['skill_items'] = [
        [] if pd.isna(value) else [str(value)] for value in skill_item.to_numpy()
    ]
    return raw[
        ['learner_id', 'event_id', 'event_time', 'problem_item', 'skill_items']
    ].copy()

def load_kdd_ordered_events():
    columns = [
        'Row', 'Anon Student Id', 'Correct First Attempt',
        'Step Start Time', 'First Transaction Time', 'Step End Time',
        'KC(Default)', 'Problem Hierarchy', 'Problem Name',
    ]
    raw = pd.read_csv(
        KDD_RAW_PATH, sep='\t', usecols=columns, low_memory=False
    )
    for column in ['Row', 'Correct First Attempt']:
        raw[column] = pd.to_numeric(raw[column], errors='coerce')
    for column in [
        'Step Start Time', 'First Transaction Time', 'Step End Time'
    ]:
        raw[column] = pd.to_datetime(raw[column], errors='coerce')
    raw['event_time'] = (
        raw['Step Start Time']
        .fillna(raw['First Transaction Time'])
        .fillna(raw['Step End Time'])
    )
    duplicate = raw['Row'].notna() & raw.duplicated('Row', keep='first')
    raw = raw.loc[~duplicate]
    raw = raw[
        raw['Anon Student Id'].notna()
        & raw['Correct First Attempt'].isin([0, 1])
    ].copy()
    raw['learner_id'] = normalise_sequence_learner_ids(
        raw['Anon Student Id']
    )
    raw['event_id'] = raw['Row']
    raw['skill_items'] = raw['KC(Default)'].fillna('').astype(str).apply(
        lambda value: [
            f'skill:{part.strip()}'
            for part in value.split('~~') if part.strip()
        ]
    )
    raw['problem_item'] = (
        'problem:'
        + raw['Problem Hierarchy'].fillna('Unknown').astype(str).str.strip()
        + '::'
        + raw['Problem Name'].fillna('Unknown').astype(str).str.strip()
    )
    return raw[
        ['learner_id', 'event_id', 'event_time', 'problem_item', 'skill_items']
    ].copy()

def reconstruct_frozen_early_events(dataset):
    learners = pd.read_parquet(
        DATASET_DIRS[dataset] / 'learner_splits.parquet'
    )
    frozen_ids = set(learners['learner_id'].astype(str))
    cohort_lookup = learners.set_index('learner_id')['cohort']
    events = (
        load_assist_ordered_events()
        if dataset == 'ASSISTments' else load_kdd_ordered_events()
    )
    events = events[events['learner_id'].astype(str).isin(frozen_ids)].copy()
    events = events.sort_values(
        ['learner_id', 'event_time', 'event_id'],
        na_position='last', kind='mergesort',
    )
    events['event_number'] = events.groupby('learner_id').cumcount()
    events['learner_event_count'] = events.groupby('learner_id')[
        'learner_id'
    ].transform('size')
    raw_cutoff = np.floor(
        events['learner_event_count'] * EARLY_FRACTION
    ).astype(int)
    events['split_point'] = np.minimum(
        np.maximum(raw_cutoff, 1), events['learner_event_count'] - 1
    )
    early = events[events['event_number'] < events['split_point']].copy()
    early['cohort'] = early['learner_id'].map(cohort_lookup)
    reconstructed_counts = early.groupby('learner_id').size().sort_index()
    frozen_counts = learners.set_index('learner_id')[
        'early_interactions'
    ].sort_index().astype('int64')
    assert reconstructed_counts.index.equals(frozen_counts.index)
    assert reconstructed_counts.astype('int64').equals(frozen_counts)
    assert set(early['cohort']) == {'discovery', 'validation'}
    return early, learners

ordered_early_events = {}
sequence_reconstruction_rows = []
for dataset in DATASET_DIRS:
    early_events, learners = reconstruct_frozen_early_events(dataset)
    ordered_early_events[dataset] = early_events
    sequence_reconstruction_rows.append({
        'Dataset': dataset,
        'ReconstructedEarlyEvents': len(early_events),
        'FrozenEarlyInteractions': int(learners['early_interactions'].sum()),
        'Learners': learners['learner_id'].nunique(),
        'DiscoveryLearners': learners['cohort'].eq('discovery').sum(),
        'ValidationLearners': learners['cohort'].eq('validation').sum(),
        'LearnerLevelCountMatch': True,
    })
sequence_reconstruction = pd.DataFrame(sequence_reconstruction_rows)
display(sequence_reconstruction)

,Dataset,ReconstructedEarlyEvents,FrozenEarlyInteractions,Learners,DiscoveryLearners,ValidationLearners,LearnerLevelCountMatch
0,ASSISTments,4184236,4184236,33335,26668,6667,True
1,KDD,566460,566460,565,452,113,True


In [20]:
def event_item_lists(events, task, catalog_items):
    frame = events[
        ['learner_id', 'event_id', 'event_number', 'cohort',
         'problem_item', 'skill_items']
    ].copy()
    if task == 'problem':
        frame['item_list'] = frame['problem_item'].apply(
            lambda value: (
                [str(value)]
                if pd.notna(value) and str(value) in catalog_items else []
            )
        )
    elif task == 'skill':
        frame['item_list'] = frame['skill_items'].apply(
            lambda values: [
                str(value) for value in values if str(value) in catalog_items
            ]
        )
    else:
        raise ValueError(f'Unknown task: {task}')
    return frame.sort_values(
        ['learner_id', 'event_number', 'event_id'], kind='mergesort'
    )

def adjacent_transition_probabilities(event_items):
    discovery = event_items[event_items['cohort'].eq('discovery')].copy()
    discovery['previous_items'] = discovery.groupby('learner_id')[
        'item_list'
    ].shift(1)
    adjacent = discovery[
        discovery['item_list'].map(bool)
        & discovery['previous_items'].map(
            lambda value: isinstance(value, list) and bool(value)
        )
    ][['learner_id', 'previous_items', 'item_list']].copy()
    transitions = (
        adjacent.explode('previous_items')
        .explode('item_list')
        .rename(columns={
            'previous_items': 'source_item_id',
            'item_list': 'target_item_id',
        })
    )
    counts = transitions.groupby(
        ['source_item_id', 'target_item_id'], sort=False
    ).size().rename('transition_count').reset_index()
    source_totals = counts.groupby('source_item_id')[
        'transition_count'
    ].transform('sum')
    counts['source_transition_total'] = source_totals
    counts['transition_probability'] = (
        counts['transition_count'] / source_totals
    )
    counts['transition_source'] = 'discovery_early_adjacent_events_only'
    assert np.allclose(
        counts.groupby('source_item_id')['transition_probability'].sum(), 1.0
    )
    return counts

def recent_sources_for_cohort(
    event_items, cohort, recent_items=RECENT_SOURCE_ITEMS,
):
    eligible = event_items['cohort'].eq(cohort) & event_items['item_list'].map(bool)
    eligible_learners = event_items.loc[eligible, 'learner_id']
    reverse_event_rank = eligible_learners.groupby(
        eligible_learners, sort=False
    ).cumcount(ascending=False)
    recent_event_index = eligible_learners.index[
        reverse_event_rank < recent_items
    ]
    stream = event_items.loc[
        recent_event_index,
        ['learner_id', 'event_number', 'event_id', 'item_list'],
    ].explode('item_list').rename(columns={'item_list': 'source_item_id'})
    stream = stream[stream['source_item_id'].notna()].copy()
    stream = stream.sort_values(
        ['learner_id', 'event_number', 'event_id', 'source_item_id'],
        kind='mergesort',
    )
    stream['recency_rank'] = (
        stream.groupby('learner_id').cumcount(ascending=False)
    )
    return stream[stream['recency_rank'] < recent_items][
        ['learner_id', 'source_item_id', 'recency_rank']
    ].copy()

transition_parts = []
validation_recent_source_parts = []
discovery_recent_source_parts = []
sequence_diagnostic_rows = []
for task_definition in TASK_DEFINITIONS:
    dataset = task_definition['Dataset']
    task = task_definition['Task']
    catalog = pd.read_parquet(
        DATASET_DIRS[dataset] / task_definition['CatalogFile']
    )
    catalog_items = set(catalog['item_id'].astype(str))
    event_items = event_item_lists(
        ordered_early_events[dataset], task, catalog_items
    )
    transitions = adjacent_transition_probabilities(event_items)
    recent_sources = recent_sources_for_cohort(event_items, 'validation')
    discovery_recent_sources = recent_sources_for_cohort(event_items, 'discovery')
    assert set(transitions['source_item_id']).issubset(catalog_items)
    assert set(transitions['target_item_id']).issubset(catalog_items)
    assert set(recent_sources['source_item_id']).issubset(catalog_items)
    assert set(discovery_recent_sources['source_item_id']).issubset(catalog_items)
    transitions.insert(0, 'Task', task)
    transitions.insert(0, 'Dataset', dataset)
    recent_sources.insert(0, 'Task', task)
    recent_sources.insert(0, 'Dataset', dataset)
    discovery_recent_sources.insert(0, 'Task', task)
    discovery_recent_sources.insert(0, 'Dataset', dataset)
    transition_parts.append(transitions)
    validation_recent_source_parts.append(recent_sources)
    discovery_recent_source_parts.append(discovery_recent_sources)
    sequence_diagnostic_rows.append({
        'Dataset': dataset,
        'Task': task,
        'TransitionPairs': len(transitions),
        'TransitionSourceItems': transitions['source_item_id'].nunique(),
        'TransitionTargetItems': transitions['target_item_id'].nunique(),
        'TransitionEvents': int(transitions['transition_count'].sum()),
        'ValidationLearnersWithRecentSources': recent_sources['learner_id'].nunique(),
        'RecentSourceRows': len(recent_sources),
        'DiscoveryLearnersWithRecentSources': discovery_recent_sources['learner_id'].nunique(),
        'DiscoveryRecentSourceRows': len(discovery_recent_sources),
        'TrainingCohort': 'discovery',
        'TrainingWindow': 'early',
        'AdjacentEventsOnly': True,
    })

transition_probabilities = pd.concat(transition_parts, ignore_index=True)
validation_recent_items = pd.concat(
    validation_recent_source_parts, ignore_index=True
)
discovery_recent_items = pd.concat(
    discovery_recent_source_parts, ignore_index=True
)
sequence_diagnostics = pd.DataFrame(sequence_diagnostic_rows)
del ordered_early_events
gc.collect()
display(sequence_diagnostics)

,Dataset,Task,TransitionPairs,TransitionSourceItems,TransitionTargetItems,TransitionEvents,ValidationLearnersWithRecentSources,RecentSourceRows,DiscoveryLearnersWithRecentSources,DiscoveryRecentSourceRows,TrainingCohort,TrainingWindow,AdjacentEventsOnly
0,ASSISTments,problem,782351,39602,39669,2699709,6503,19458,26058,78018,discovery,early,True
1,ASSISTments,skill,10774,161,161,1355028,4162,12354,16453,48859,discovery,early,True
2,KDD,problem,7405,855,855,460379,113,339,451,1353,discovery,early,True
3,KDD,skill,2139,94,98,653284,113,339,451,1353,discovery,early,True


## Step 12 — Sequential recommendation baseline

The baseline combines transitions from each learner's three most recent supported early items using exponential recency decay. Positive transition scores rank first. If fewer than 20 transition candidates are available—or a recent source has no learned outgoing transition—the remaining positions are filled by discovery-early popularity.


In [21]:
SEQUENTIAL_RECENCY_DECAY = 0.70

def recency_weighted_transition_scores(
    recent_sources, transitions, recency_decay=SEQUENTIAL_RECENCY_DECAY,
):
    joined = recent_sources.merge(
        transitions[
            ['source_item_id', 'target_item_id', 'transition_probability']
        ],
        on='source_item_id', how='inner', validate='many_to_many',
    )
    joined['recency_weight'] = np.power(
        recency_decay, joined['recency_rank'].astype(float)
    )
    joined['weighted_transition_score'] = (
        joined['recency_weight'] * joined['transition_probability']
    )
    scores = joined.groupby(
        ['learner_id', 'target_item_id'], sort=False
    )['weighted_transition_score'].sum().rename('score').reset_index().rename(
        columns={'target_item_id': 'item_id'}
    )
    return scores

def sequential_top_k_with_popularity_fallback(
    validation_learner_ids, sequential_scores, catalog, early_history,
    candidate_policy, max_k=MAX_RECOMMENDATIONS,
):
    if candidate_policy not in CANDIDATE_POLICIES:
        raise ValueError(f'Unknown candidate policy: {candidate_policy}')
    learner_ids = sorted(pd.Series(validation_learner_ids).astype(str).unique())
    policy_scores = apply_candidate_policy(
        sequential_scores, early_history, candidate_policy
    )
    sequential_by_learner = {
        learner_id: list(
            group.sort_values(
                ['score', 'item_id'], ascending=[False, True], kind='mergesort'
            )[['item_id', 'score']].itertuples(index=False, name=None)
        )
        for learner_id, group in policy_scores.groupby('learner_id', sort=False)
    }
    popularity_ranking = list(
        catalog[['item_id', 'training_interactions']]
        .assign(score=lambda frame: frame['training_interactions'].astype(float))
        .sort_values(['score', 'item_id'], ascending=[False, True], kind='mergesort')
        [['item_id', 'score']].itertuples(index=False, name=None)
    )
    seen_by_learner = {}
    if candidate_policy == 'novel_only':
        learner_set = set(learner_ids)
        seen = early_history[
            early_history['learner_id'].astype(str).isin(learner_set)
            & early_history['in_candidate_catalog']
        ][['learner_id', 'item_id']].drop_duplicates()
        seen_by_learner = seen.groupby('learner_id')['item_id'].agg(set).to_dict()

    rows = []
    for learner_id in learner_ids:
        selected = set()
        rank = 0
        for item_id, score in sequential_by_learner.get(learner_id, []):
            if item_id in selected:
                continue
            rank += 1
            selected.add(item_id)
            rows.append((
                learner_id, item_id, float(score), rank, 'transition'
            ))
            if rank == max_k:
                break
        if rank < max_k:
            excluded = seen_by_learner.get(learner_id, set())
            for popularity_position, (item_id, _) in enumerate(
                popularity_ranking, start=1
            ):
                if item_id in selected or item_id in excluded:
                    continue
                rank += 1
                selected.add(item_id)
                fallback_score = -float(popularity_position)
                rows.append((
                    learner_id, item_id, fallback_score, rank, 'popularity_fallback'
                ))
                if rank == max_k:
                    break
    recommendations = pd.DataFrame(rows, columns=[
        'learner_id', 'item_id', 'score', 'rank', 'score_source'
    ])
    assert not recommendations.duplicated(['learner_id', 'item_id']).any()
    assert recommendations.groupby('learner_id').size().le(max_k).all()
    assert np.isfinite(recommendations['score']).all()
    assert_candidate_subset(recommendations, catalog)
    return recommendations

In [22]:
SEQUENTIAL_RECENCY_CANDIDATES = (0.50, 0.70, 0.90)
SEQUENTIAL_SHRINKAGE_CANDIDATES = (0.0, 10.0)
assert SEQUENTIAL_RECENCY_DECAY in SEQUENTIAL_RECENCY_CANDIDATES

## Save and verify Steps 11-12 artifacts


In [23]:
steps_11_12_paths = {
    'transition_probabilities': OUTPUT_ROOT / 'step11_transition_probabilities.parquet',
    'validation_recent_items': OUTPUT_ROOT / 'step11_validation_recent_items.parquet',
    'discovery_recent_items': OUTPUT_ROOT / 'step11_discovery_recent_items.parquet',
    'sequence_diagnostics': OUTPUT_ROOT / 'step11_sequence_diagnostics.csv',
    'config': OUTPUT_ROOT / 'steps_11_12_config.json',
}
transition_probabilities.to_parquet(
    steps_11_12_paths['transition_probabilities'],
    index=False, compression='snappy',
)
validation_recent_items.to_parquet(
    steps_11_12_paths['validation_recent_items'],
    index=False, compression='snappy',
)
discovery_recent_items.to_parquet(
    steps_11_12_paths['discovery_recent_items'],
    index=False, compression='snappy',
)
sequence_diagnostics.to_csv(
    steps_11_12_paths['sequence_diagnostics'], index=False
)
steps_11_12_config = {
    'implemented_steps': [11, 12],
    'raw_paths': {
        'ASSISTments': str(ASSIST_RAW_PATH), 'KDD': str(KDD_RAW_PATH),
    },
    'early_fraction': EARLY_FRACTION,
    'cohort_source': 'frozen_phase2_learner_splits',
    'assist_skill_mapping_source': 'frozen_discovery_early_phase2_map',
    'transition_training_cohort': 'discovery',
    'transition_training_window': 'early',
    'transition_definition': 'adjacent_supported_raw_events',
    'recent_source_items': RECENT_SOURCE_ITEMS,
    'default_recency_decay': SEQUENTIAL_RECENCY_DECAY,
    'recency_decay_candidates': list(SEQUENTIAL_RECENCY_CANDIDATES),
    'transition_shrinkage_candidates': list(SEQUENTIAL_SHRINKAGE_CANDIDATES),
    'fallback': 'discovery_early_popularity',
    'candidate_policies': list(CANDIDATE_POLICIES),
    'relevance_definitions': RELEVANCE_DEFINITIONS,
    'validation_evaluation_deferred_until_discovery_selection': True,
    'validation_future_role': 'evaluation_only',
    'dense_learner_item_matrix_constructed': False,
}
with open(steps_11_12_paths['config'], 'w', encoding='utf-8') as file:
    json.dump(steps_11_12_config, file, indent=2)

steps_11_12_manifest = pd.DataFrame([
    {
        'Artifact': name,
        'File': path.name,
        'Rows': (
            pq.ParquetFile(path).metadata.num_rows
            if path.suffix == '.parquet'
            else (len(pd.read_csv(path)) if path.suffix == '.csv' else 1)
        ),
        'Bytes': path.stat().st_size,
    }
    for name, path in steps_11_12_paths.items()
])
steps_11_12_manifest.to_csv(
    OUTPUT_ROOT / 'steps_11_12_artifact_manifest.csv', index=False
)
assert len(steps_11_12_manifest) == 5
assert pq.ParquetFile(
    steps_11_12_paths['transition_probabilities']
).metadata.num_rows == len(transition_probabilities)
display(steps_11_12_manifest)

,Artifact,File,Rows,Bytes
0,transition_probabilities,step11_transition_probabilities.parquet,802669,6621768
1,validation_recent_items,step11_validation_recent_items.parquet,32490,203729
2,discovery_recent_items,step11_discovery_recent_items.parquet,129583,714680
3,sequence_diagnostics,step11_sequence_diagnostics.csv,4,564
4,config,steps_11_12_config.json,1,1109


## Step 13 — Discovery-only hyperparameter selection

Step 9 selected neighbour count on a discovery-training/discovery-tuning split. This section selects neighbour weighting, content difficulty tolerance, and sequential transition shrinkage/recency decay using discovery data only. Each selected model is then refitted on all discovery learners and evaluated once on validation learners.


In [24]:
def score_content_components(components, difficulty_tolerance):
    scored = components.copy()
    scored['difficulty_suitability'] = (
        1.0 - (scored['difficulty_proxy'] - scored['profile_mastery']).abs()
        / float(difficulty_tolerance)
    ).clip(0, 1)
    scored['score'] = (
        CONTENT_WEIGHTS['weak_skill_alignment']
        * scored['weak_skill_alignment'].clip(0, 1)
        + CONTENT_WEIGHTS['difficulty_suitability'] * scored['difficulty_suitability']
        + CONTENT_WEIGHTS['problem_type_affinity'] * scored['problem_type_affinity']
        + CONTENT_WEIGHTS['hierarchy_affinity'] * scored['hierarchy_affinity']
        + POPULARITY_TIE_BREAK_WEIGHT * scored['training_popularity']
    )
    assert np.isfinite(scored['score']).all()
    return scored

content_tuning_parts = []
content_selection_rows = []
selected_content_metric_parts = []
selected_content_recommendation_parts = []
selected_content_score_parts = []
for task_definition in [row for row in TASK_DEFINITIONS if row['Task'] == 'problem']:
    dataset = task_definition['Dataset']
    tables = load_task_tables(task_definition)
    discovery_ids = sorted(tables['learners'].loc[
        tables['learners']['cohort'].eq('discovery'), 'learner_id'
    ].astype(str))
    validation_ids = sorted(tables['learners'].loc[
        tables['learners']['cohort'].eq('validation'), 'learner_id'
    ].astype(str))
    assert set(discovery_ids).isdisjoint(validation_ids)
    _, content_tuning_ids = train_test_split(
        discovery_ids, test_size=DISCOVERY_TUNING_FRACTION,
        random_state=RANDOM_STATE,
    )
    content_tuning_ids = sorted(content_tuning_ids)
    assert set(content_tuning_ids).issubset(discovery_ids)
    assert set(content_tuning_ids).isdisjoint(validation_ids)
    discovery_relevance = tables['future_relevance'][
        tables['future_relevance']['learner_id'].astype(str).isin(content_tuning_ids)
    ].copy()
    assert set(discovery_relevance['learner_id'].astype(str)).isdisjoint(validation_ids)
    discovery_components, catalog = content_components_for_learners(
        task_definition, content_tuning_ids
    )
    task_tuning = []
    for tolerance in CONTENT_DIFFICULTY_TOLERANCES:
        scores = score_content_components(discovery_components, tolerance)
        recommendations = top_k_sparse(scores, MAX_RECOMMENDATIONS)
        metrics = evaluate_recommendations(
            recommendations, discovery_relevance, catalog, tables['learners'],
            RELEVANCE_DEFINITIONS['attempted'], task_definition['EvaluableColumn'],
            task_definition['ColdStartColumn'], dataset, 'problem',
            'all_supported', 'attempted', ks=(10,),
            evaluation_learner_ids=content_tuning_ids,
        )
        metrics.insert(4, 'Model', 'content_problem_tuning')
        metrics['DifficultyTolerance'] = tolerance
        metrics['TuningSource'] = 'discovery_only'
        task_tuning.append(metrics)
        content_tuning_parts.append(metrics)
    candidates = pd.concat(task_tuning, ignore_index=True)
    candidates = candidates[candidates['Segment'].eq('all')].sort_values(
        ['NDCGAtK', 'DifficultyTolerance'], ascending=[False, True],
        kind='mergesort',
    )
    selected_tolerance = float(candidates.iloc[0]['DifficultyTolerance'])
    content_selection_rows.append({
        'Dataset': dataset, 'Task': 'problem',
        'SelectedDifficultyTolerance': selected_tolerance,
        'SelectionMetric': 'NDCGAtK', 'SelectionK': 10,
        'SelectionRelevance': 'attempted',
        'SelectionCandidatePolicy': 'all_supported',
        'DiscoveryTuningLearners': len(content_tuning_ids),
        'ValidationLearnersUntouched': len(validation_ids),
    })
    validation_components, catalog = content_components_for_learners(
        task_definition, validation_ids
    )
    final_scores = score_content_components(validation_components, selected_tolerance)
    score_output = final_scores.copy()
    score_output.insert(0, 'Task', 'problem')
    score_output.insert(0, 'Dataset', dataset)
    score_output['DifficultyTolerance'] = selected_tolerance
    selected_content_score_parts.append(score_output)
    for candidate_policy in CANDIDATE_POLICIES:
        policy_scores = apply_candidate_policy(
            final_scores, tables['early_history'], candidate_policy
        )
        recommendations = top_k_sparse(policy_scores, MAX_RECOMMENDATIONS)
        assert_candidate_subset(recommendations, catalog)
        for relevance_name, relevance_column in RELEVANCE_DEFINITIONS.items():
            metrics = evaluate_recommendations(
                recommendations, tables['future_relevance'], catalog, tables['learners'],
                relevance_column, task_definition['EvaluableColumn'],
                task_definition['ColdStartColumn'], dataset, 'problem',
                candidate_policy, relevance_name,
            )
            metrics.insert(4, 'Model', 'content_problem')
            metrics['DifficultyTolerance'] = selected_tolerance
            selected_content_metric_parts.append(metrics)
            output = recommendations.copy()
            output.insert(0, 'RelevanceDefinition', relevance_name)
            output.insert(0, 'CandidatePolicy', candidate_policy)
            output.insert(0, 'Model', 'content_problem')
            output.insert(0, 'Task', 'problem')
            output.insert(0, 'Dataset', dataset)
            output['DifficultyTolerance'] = selected_tolerance
            selected_content_recommendation_parts.append(output)

content_tuning_metrics = pd.concat(content_tuning_parts, ignore_index=True)
content_selection = pd.DataFrame(content_selection_rows)
selected_content_metrics = pd.concat(selected_content_metric_parts, ignore_index=True)
selected_content_recommendations = pd.concat(
    selected_content_recommendation_parts, ignore_index=True
)
selected_content_scores = pd.concat(selected_content_score_parts, ignore_index=True)
display(content_selection)
display(content_tuning_metrics.sort_values(['Dataset', 'DifficultyTolerance', 'Segment']))

NEIGHBOR_WEIGHTING_CANDIDATES = ('cosine', 'uniform_positive')
neighbor_weighting_tuning_parts = []
neighbor_weighting_selection_rows = []
selected_neighbor_metric_parts = []
selected_neighbor_recommendation_parts = []
for task_definition in TASK_DEFINITIONS:
    dataset = task_definition['Dataset']
    task = task_definition['Task']
    tables = load_task_tables(task_definition)
    item_order = sorted(tables['catalog']['item_id'].astype(str).unique())
    discovery_ids = sorted(tables['learners'].loc[
        tables['learners']['cohort'].eq('discovery'), 'learner_id'
    ].astype(str))
    validation_ids = sorted(tables['learners'].loc[
        tables['learners']['cohort'].eq('validation'), 'learner_id'
    ].astype(str))
    discovery_train_ids, discovery_tuning_ids = train_test_split(
        discovery_ids, test_size=DISCOVERY_TUNING_FRACTION, random_state=RANDOM_STATE
    )
    discovery_train_ids = sorted(discovery_train_ids)
    discovery_tuning_ids = sorted(discovery_tuning_ids)
    validation_set = set(validation_ids)
    assert set(discovery_train_ids).isdisjoint(validation_set)
    assert set(discovery_tuning_ids).isdisjoint(validation_set)
    discovery_tuning_relevance = tables['future_relevance'][
        tables['future_relevance']['learner_id'].astype(str).isin(discovery_tuning_ids)
    ].copy()
    selected_neighbors = int(neighbor_selection.loc[
        neighbor_selection['Dataset'].eq(dataset)
        & neighbor_selection['Task'].eq(task), 'SelectedNeighbors'
    ].iloc[0])
    training_matrix = build_sparse_history_matrix(
        tables['early_history'], discovery_train_ids, item_order
    )
    tuning_matrix = build_sparse_history_matrix(
        tables['early_history'], discovery_tuning_ids, item_order
    )
    neighbor_indices, cosine_similarities = query_sparse_neighbors(
        training_matrix, tuning_matrix, selected_neighbors
    )
    target_matrix = build_discovery_target_matrix(
        tables['future_relevance'], discovery_train_ids, item_order,
        RELEVANCE_DEFINITIONS['attempted'],
    )
    task_tuning = []
    for weighting in NEIGHBOR_WEIGHTING_CANDIDATES:
        similarities = (
            cosine_similarities if weighting == 'cosine'
            else (cosine_similarities > 0).astype(np.float32)
        )
        recommendations = aggregate_neighbor_recommendations(
            discovery_tuning_ids, target_matrix, neighbor_indices, similarities,
            item_order, tables['early_history'], 'all_supported',
        )
        metrics = evaluate_recommendations(
            recommendations, discovery_tuning_relevance, tables['catalog'],
            tables['learners'], RELEVANCE_DEFINITIONS['attempted'],
            task_definition['EvaluableColumn'], task_definition['ColdStartColumn'],
            dataset, task, 'all_supported', 'attempted', ks=(10,),
            evaluation_learner_ids=discovery_tuning_ids,
        )
        metrics.insert(4, 'Model', 'learner_neighbor_cf_weighting_tuning')
        metrics['NeighborCount'] = selected_neighbors
        metrics['NeighborWeighting'] = weighting
        metrics['TuningSource'] = 'discovery_only'
        task_tuning.append(metrics)
        neighbor_weighting_tuning_parts.append(metrics)
    candidates = pd.concat(task_tuning, ignore_index=True)
    candidates['WeightingTieOrder'] = candidates['NeighborWeighting'].map({
        'cosine': 0, 'uniform_positive': 1
    })
    candidates = candidates[candidates['Segment'].eq('all')].sort_values(
        ['NDCGAtK', 'WeightingTieOrder'], ascending=[False, True],
        kind='mergesort',
    )
    selected_weighting = str(candidates.iloc[0]['NeighborWeighting'])
    neighbor_weighting_selection_rows.append({
        'Dataset': dataset, 'Task': task, 'SelectedNeighbors': selected_neighbors,
        'SelectedNeighborWeighting': selected_weighting,
        'SelectionMetric': 'NDCGAtK', 'SelectionK': 10,
        'SelectionRelevance': 'attempted',
        'SelectionCandidatePolicy': 'all_supported',
        'DiscoveryTrainingLearners': len(discovery_train_ids),
        'DiscoveryTuningLearners': len(discovery_tuning_ids),
        'ValidationLearnersUntouched': len(validation_ids),
    })
    final_training_matrix = build_sparse_history_matrix(
        tables['early_history'], discovery_ids, item_order
    )
    validation_matrix = build_sparse_history_matrix(
        tables['early_history'], validation_ids, item_order
    )
    final_indices, final_cosine_similarities = query_sparse_neighbors(
        final_training_matrix, validation_matrix, selected_neighbors
    )
    final_similarities = (
        final_cosine_similarities if selected_weighting == 'cosine'
        else (final_cosine_similarities > 0).astype(np.float32)
    )
    for relevance_name, relevance_column in RELEVANCE_DEFINITIONS.items():
        final_target_matrix = build_discovery_target_matrix(
            tables['future_relevance'], discovery_ids, item_order, relevance_column
        )
        for candidate_policy in CANDIDATE_POLICIES:
            recommendations = aggregate_neighbor_recommendations(
                validation_ids, final_target_matrix, final_indices, final_similarities,
                item_order, tables['early_history'], candidate_policy,
            )
            metrics = evaluate_recommendations(
                recommendations, tables['future_relevance'], tables['catalog'],
                tables['learners'], relevance_column,
                task_definition['EvaluableColumn'], task_definition['ColdStartColumn'],
                dataset, task, candidate_policy, relevance_name,
            )
            metrics.insert(4, 'Model', 'learner_neighbor_cf')
            metrics['NeighborCount'] = selected_neighbors
            metrics['NeighborWeighting'] = selected_weighting
            metrics['TrainingLearnerCohort'] = 'discovery'
            selected_neighbor_metric_parts.append(metrics)
            output = recommendations.copy()
            output.insert(0, 'RelevanceDefinition', relevance_name)
            output.insert(0, 'CandidatePolicy', candidate_policy)
            output.insert(0, 'Model', 'learner_neighbor_cf')
            output.insert(0, 'Task', task)
            output.insert(0, 'Dataset', dataset)
            output['NeighborCount'] = selected_neighbors
            output['NeighborWeighting'] = selected_weighting
            selected_neighbor_recommendation_parts.append(output)

neighbor_weighting_tuning_metrics = pd.concat(
    neighbor_weighting_tuning_parts, ignore_index=True
)
selected_neighbor_configuration = pd.DataFrame(neighbor_weighting_selection_rows)
selected_neighbor_metrics = pd.concat(selected_neighbor_metric_parts, ignore_index=True)
selected_neighbor_recommendations = pd.concat(
    selected_neighbor_recommendation_parts, ignore_index=True
)
display(selected_neighbor_configuration)
display(neighbor_weighting_tuning_metrics.sort_values([
    'Dataset', 'Task', 'NeighborWeighting', 'Segment'
]))

step13_model_paths = {
    'content_tuning': OUTPUT_ROOT / 'step13_content_tuning.csv',
    'content_selection': OUTPUT_ROOT / 'step13_content_selection.csv',
    'content_metrics': OUTPUT_ROOT / 'step13_selected_content_metrics.csv',
    'content_recommendations': OUTPUT_ROOT / 'step13_selected_content_recommendations.parquet',
    'content_scores': OUTPUT_ROOT / 'step13_selected_content_scores.parquet',
    'neighbor_weighting_tuning': OUTPUT_ROOT / 'step13_neighbor_weighting_tuning.csv',
    'neighbor_selection': OUTPUT_ROOT / 'step13_selected_neighbor_configuration.csv',
    'neighbor_metrics': OUTPUT_ROOT / 'step13_selected_neighbor_metrics.csv',
    'neighbor_recommendations': OUTPUT_ROOT / 'step13_selected_neighbor_recommendations.parquet',
}
content_tuning_metrics.to_csv(step13_model_paths['content_tuning'], index=False)
content_selection.to_csv(step13_model_paths['content_selection'], index=False)
selected_content_metrics.to_csv(step13_model_paths['content_metrics'], index=False)
selected_content_recommendations.to_parquet(
    step13_model_paths['content_recommendations'], index=False, compression='snappy'
)
selected_content_scores.to_parquet(
    step13_model_paths['content_scores'], index=False, compression='snappy'
)
neighbor_weighting_tuning_metrics.to_csv(
    step13_model_paths['neighbor_weighting_tuning'], index=False
)
selected_neighbor_configuration.to_csv(
    step13_model_paths['neighbor_selection'], index=False
)
selected_neighbor_metrics.to_csv(step13_model_paths['neighbor_metrics'], index=False)
selected_neighbor_recommendations.to_parquet(
    step13_model_paths['neighbor_recommendations'], index=False, compression='snappy'
)
for large_name in [
    'discovery_components', 'validation_components', 'final_scores',
    'score_output', 'selected_content_scores',
    'selected_content_recommendations', 'training_matrix', 'tuning_matrix',
    'final_training_matrix', 'validation_matrix', 'target_matrix',
    'final_target_matrix', 'selected_neighbor_recommendations',
    'recommendations', 'neighbor_indices', 'cosine_similarities',
    'final_indices', 'final_cosine_similarities',
]:
    globals().pop(large_name, None)
_ = gc.collect()


,Dataset,Task,SelectedDifficultyTolerance,SelectionMetric,SelectionK,SelectionRelevance,SelectionCandidatePolicy,DiscoveryTuningLearners,ValidationLearnersUntouched
0,ASSISTments,problem,1.0,NDCGAtK,10,attempted,all_supported,5334,6667
1,KDD,problem,1.0,NDCGAtK,10,attempted,all_supported,91,113


,Dataset,Task,CandidatePolicy,RelevanceDefinition,Model,Segment,K,PrecisionAtK,RecallAtK,NDCGAtK,...,HitRateAtK,CatalogCoverageAtK,MeanRecommendations,EvaluatedLearners,EvaluableRate,EvaluationScope,RecommendationMemoryBytes,MetricRuntimeSeconds,DifficultyTolerance,TuningSource
0,ASSISTments,problem,all_supported,attempted,content_problem_tuning,all,10,0.006335,0.002656,0.006901,...,0.046676,0.165942,5.995624,4799,0.899700,provided_learner_ids,8526793,0.958345,0.25,discovery_only
1,ASSISTments,problem,all_supported,attempted,content_problem_tuning,cold_start,10,0.000000,0.000000,0.000000,...,0.000000,0.000754,2.307692,13,0.899700,provided_learner_ids,8526793,0.958345,0.25,discovery_only
2,ASSISTments,problem,all_supported,attempted,content_problem_tuning,non_cold_start,10,0.006352,0.002663,0.006919,...,0.046803,0.165866,6.005641,4786,0.899700,provided_learner_ids,8526793,0.958345,0.25,discovery_only
3,ASSISTments,problem,all_supported,attempted,content_problem_tuning,all,10,0.006981,0.003250,0.007717,...,0.048969,0.160135,5.995624,4799,0.899700,provided_learner_ids,8526573,0.988961,0.50,discovery_only
4,ASSISTments,problem,all_supported,attempted,content_problem_tuning,cold_start,10,0.000000,0.000000,0.000000,...,0.000000,0.000754,2.307692,13,0.899700,provided_learner_ids,8526573,0.988961,0.50,discovery_only
5,ASSISTments,problem,all_supported,attempted,content_problem_tuning,non_cold_start,10,0.007000,0.003259,0.007737,...,0.049102,0.160034,6.005641,4786,0.899700,provided_learner_ids,8526573,0.988961,0.50,discovery_only
6,ASSISTments,problem,all_supported,attempted,content_problem_tuning,all,10,0.007335,0.003372,0.008272,...,0.049385,0.158652,5.995624,4799,0.899700,provided_learner_ids,8526345,1.566130,1.00,discovery_only
7,ASSISTments,problem,all_supported,attempted,content_problem_tuning,cold_start,10,0.000000,0.000000,0.000000,...,0.000000,0.000754,2.307692,13,0.899700,provided_learner_ids,8526345,1.566130,1.00,discovery_only
8,ASSISTments,problem,all_supported,attempted,content_problem_tuning,non_cold_start,10,0.007355,0.003382,0.008294,...,0.049519,0.158551,6.005641,4786,0.899700,provided_learner_ids,8526345,1.566130,1.00,discovery_only
9,KDD,problem,all_supported,attempted,content_problem_tuning,all,10,0.042697,0.056929,0.061999,...,0.224719,0.270175,10.000000,89,0.978022,provided_learner_ids,329898,0.046966,0.25,discovery_only


,Dataset,Task,SelectedNeighbors,SelectedNeighborWeighting,SelectionMetric,SelectionK,SelectionRelevance,SelectionCandidatePolicy,DiscoveryTrainingLearners,DiscoveryTuningLearners,ValidationLearnersUntouched
0,ASSISTments,problem,20,cosine,NDCGAtK,10,attempted,all_supported,21334,5334,6667
1,ASSISTments,skill,20,cosine,NDCGAtK,10,attempted,all_supported,21334,5334,6667
2,KDD,problem,20,cosine,NDCGAtK,10,attempted,all_supported,361,91,113
3,KDD,skill,20,cosine,NDCGAtK,10,attempted,all_supported,361,91,113


,Dataset,Task,CandidatePolicy,RelevanceDefinition,Model,Segment,K,PrecisionAtK,RecallAtK,NDCGAtK,...,CatalogCoverageAtK,MeanRecommendations,EvaluatedLearners,EvaluableRate,EvaluationScope,RecommendationMemoryBytes,MetricRuntimeSeconds,NeighborCount,NeighborWeighting,TuningSource
0,ASSISTments,problem,all_supported,attempted,learner_neighbor_cf_weighting_tuning,all,10,0.440946,0.265667,0.498437,...,0.194474,9.802667,4799,0.899700,provided_learner_ids,13023370,1.448426,20,cosine,discovery_only
1,ASSISTments,problem,all_supported,attempted,learner_neighbor_cf_weighting_tuning,cold_start,10,0.000000,0.000000,0.000000,...,0.000000,0.000000,13,0.899700,provided_learner_ids,13023370,1.448426,20,cosine,discovery_only
2,ASSISTments,problem,all_supported,attempted,learner_neighbor_cf_weighting_tuning,non_cold_start,10,0.442144,0.266388,0.499790,...,0.194474,9.829294,4786,0.899700,provided_learner_ids,13023370,1.448426,20,cosine,discovery_only
3,ASSISTments,problem,all_supported,attempted,learner_neighbor_cf_weighting_tuning,all,10,0.435091,0.259662,0.490231,...,0.184897,9.802667,4799,0.899700,provided_learner_ids,13027986,1.190872,20,uniform_positive,discovery_only
4,ASSISTments,problem,all_supported,attempted,learner_neighbor_cf_weighting_tuning,cold_start,10,0.000000,0.000000,0.000000,...,0.000000,0.000000,13,0.899700,provided_learner_ids,13027986,1.190872,20,uniform_positive,discovery_only
5,ASSISTments,problem,all_supported,attempted,learner_neighbor_cf_weighting_tuning,non_cold_start,10,0.436272,0.260367,0.491562,...,0.184897,9.829294,4786,0.899700,provided_learner_ids,13027986,1.190872,20,uniform_positive,discovery_only
6,ASSISTments,skill,all_supported,attempted,learner_neighbor_cf_weighting_tuning,all,10,0.460660,0.685504,0.737441,...,0.962733,9.021161,3119,0.584739,provided_learner_ids,6720201,0.676349,20,cosine,discovery_only
7,ASSISTments,skill,all_supported,attempted,learner_neighbor_cf_weighting_tuning,cold_start,10,0.000000,0.000000,0.000000,...,0.000000,0.000000,110,0.584739,provided_learner_ids,6720201,0.676349,20,cosine,discovery_only
8,ASSISTments,skill,all_supported,attempted,learner_neighbor_cf_weighting_tuning,non_cold_start,10,0.477501,0.710564,0.764399,...,0.962733,9.350947,3009,0.584739,provided_learner_ids,6720201,0.676349,20,cosine,discovery_only
9,ASSISTments,skill,all_supported,attempted,learner_neighbor_cf_weighting_tuning,all,10,0.455883,0.676275,0.729056,...,0.962733,9.021161,3119,0.584739,provided_learner_ids,6720770,0.726233,20,uniform_positive,discovery_only


In [25]:
def shrink_transition_probabilities(transitions, shrinkage):
    adjusted = transitions.copy()
    adjusted['transition_probability'] = (
        adjusted['transition_count']
        / (adjusted['source_transition_total'] + float(shrinkage))
    )
    return adjusted

sequential_tuning_parts = []
sequential_selection_rows = []
selected_sequential_metric_parts = []
selected_sequential_recommendation_parts = []
for task_definition in TASK_DEFINITIONS:
    dataset = task_definition['Dataset']
    task = task_definition['Task']
    tables = load_task_tables(task_definition)
    discovery_ids = sorted(tables['learners'].loc[
        tables['learners']['cohort'].eq('discovery'), 'learner_id'
    ].astype(str))
    validation_ids = sorted(tables['learners'].loc[
        tables['learners']['cohort'].eq('validation'), 'learner_id'
    ].astype(str))
    assert set(discovery_ids).isdisjoint(validation_ids)
    _, sequence_tuning_ids = train_test_split(
        discovery_ids, test_size=DISCOVERY_TUNING_FRACTION,
        random_state=RANDOM_STATE,
    )
    sequence_tuning_ids = sorted(sequence_tuning_ids)
    assert set(sequence_tuning_ids).issubset(discovery_ids)
    assert set(sequence_tuning_ids).isdisjoint(validation_ids)
    discovery_relevance = tables['future_relevance'][
        tables['future_relevance']['learner_id'].astype(str).isin(sequence_tuning_ids)
    ].copy()
    assert set(discovery_relevance['learner_id'].astype(str)).isdisjoint(validation_ids)
    discovery_recent = discovery_recent_items[
        discovery_recent_items['Dataset'].eq(dataset)
        & discovery_recent_items['Task'].eq(task)
        & discovery_recent_items['learner_id'].astype(str).isin(sequence_tuning_ids)
    ].drop(columns=['Dataset', 'Task']).copy()
    assert set(discovery_recent['learner_id'].astype(str)).issubset(
        sequence_tuning_ids
    )
    transitions = transition_probabilities[
        transition_probabilities['Dataset'].eq(dataset)
        & transition_probabilities['Task'].eq(task)
    ].drop(columns=['Dataset', 'Task'])
    task_tuning = []
    for shrinkage in SEQUENTIAL_SHRINKAGE_CANDIDATES:
        adjusted = shrink_transition_probabilities(transitions, shrinkage)
        for decay in SEQUENTIAL_RECENCY_CANDIDATES:
            scores = recency_weighted_transition_scores(
                discovery_recent, adjusted, recency_decay=decay
            )
            recommendations = sequential_top_k_with_popularity_fallback(
                sequence_tuning_ids, scores, tables['catalog'],
                tables['early_history'], 'all_supported',
            )
            metrics = evaluate_recommendations(
                recommendations, discovery_relevance, tables['catalog'],
                tables['learners'], RELEVANCE_DEFINITIONS['attempted'],
                task_definition['EvaluableColumn'], task_definition['ColdStartColumn'],
                dataset, task, 'all_supported', 'attempted', ks=(10,),
                evaluation_learner_ids=sequence_tuning_ids,
            )
            metrics.insert(4, 'Model', 'sequential_transition_tuning')
            metrics['TransitionShrinkage'] = shrinkage
            metrics['RecencyDecay'] = decay
            metrics['TuningSource'] = 'discovery_only'
            task_tuning.append(metrics)
            sequential_tuning_parts.append(metrics)
    candidates = pd.concat(task_tuning, ignore_index=True)
    candidates = candidates[candidates['Segment'].eq('all')].sort_values(
        ['NDCGAtK', 'TransitionShrinkage', 'RecencyDecay'],
        ascending=[False, True, False], kind='mergesort',
    )
    selected = candidates.iloc[0]
    selected_shrinkage = float(selected['TransitionShrinkage'])
    selected_decay = float(selected['RecencyDecay'])
    sequential_selection_rows.append({
        'Dataset': dataset, 'Task': task,
        'SelectedTransitionShrinkage': selected_shrinkage,
        'SelectedRecencyDecay': selected_decay,
        'SelectionMetric': 'NDCGAtK', 'SelectionK': 10,
        'SelectionRelevance': 'attempted',
        'SelectionCandidatePolicy': 'all_supported',
        'DiscoveryTuningLearners': len(sequence_tuning_ids),
        'ValidationLearnersUntouched': len(validation_ids),
    })
    final_transitions = shrink_transition_probabilities(transitions, selected_shrinkage)
    validation_recent = validation_recent_items[
        validation_recent_items['Dataset'].eq(dataset)
        & validation_recent_items['Task'].eq(task)
    ].drop(columns=['Dataset', 'Task'])
    final_scores = recency_weighted_transition_scores(
        validation_recent, final_transitions, recency_decay=selected_decay
    )
    for candidate_policy in CANDIDATE_POLICIES:
        recommendations = sequential_top_k_with_popularity_fallback(
            validation_ids, final_scores, tables['catalog'],
            tables['early_history'], candidate_policy,
        )
        for relevance_name, relevance_column in RELEVANCE_DEFINITIONS.items():
            metrics = evaluate_recommendations(
                recommendations, tables['future_relevance'], tables['catalog'],
                tables['learners'], relevance_column,
                task_definition['EvaluableColumn'], task_definition['ColdStartColumn'],
                dataset, task, candidate_policy, relevance_name,
            )
            metrics.insert(4, 'Model', 'sequential_transition')
            metrics['TransitionShrinkage'] = selected_shrinkage
            metrics['RecencyDecay'] = selected_decay
            metrics['FallbackModel'] = 'discovery_early_popularity'
            selected_sequential_metric_parts.append(metrics)
            output = recommendations.copy()
            output.insert(0, 'RelevanceDefinition', relevance_name)
            output.insert(0, 'CandidatePolicy', candidate_policy)
            output.insert(0, 'Model', 'sequential_transition')
            output.insert(0, 'Task', task)
            output.insert(0, 'Dataset', dataset)
            output['TransitionShrinkage'] = selected_shrinkage
            output['RecencyDecay'] = selected_decay
            selected_sequential_recommendation_parts.append(output)
    del discovery_recent
    gc.collect()

sequential_tuning_metrics = pd.concat(sequential_tuning_parts, ignore_index=True)
sequential_selection = pd.DataFrame(sequential_selection_rows)
selected_sequential_metrics = pd.concat(
    selected_sequential_metric_parts, ignore_index=True
)
selected_sequential_recommendations = pd.concat(
    selected_sequential_recommendation_parts, ignore_index=True
)
step13_model_paths.update({
    'sequential_tuning': OUTPUT_ROOT / 'step13_sequential_tuning.csv',
    'sequential_selection': OUTPUT_ROOT / 'step13_sequential_selection.csv',
    'sequential_metrics': OUTPUT_ROOT / 'step13_selected_sequential_metrics.csv',
    'sequential_recommendations': OUTPUT_ROOT / 'step13_selected_sequential_recommendations.parquet',
})
sequential_tuning_metrics.to_csv(
    step13_model_paths['sequential_tuning'], index=False
)
sequential_selection.to_csv(step13_model_paths['sequential_selection'], index=False)
selected_sequential_metrics.to_csv(
    step13_model_paths['sequential_metrics'], index=False
)
selected_sequential_recommendations.to_parquet(
    step13_model_paths['sequential_recommendations'],
    index=False, compression='snappy',
)
hyperparameter_selection = selected_neighbor_configuration[[
    'Dataset', 'Task', 'SelectedNeighbors', 'SelectedNeighborWeighting',
    'DiscoveryTrainingLearners', 'DiscoveryTuningLearners',
    'ValidationLearnersUntouched',
]].merge(
    sequential_selection[[
        'Dataset', 'Task', 'SelectedTransitionShrinkage',
        'SelectedRecencyDecay',
    ]],
    on=['Dataset', 'Task'], how='outer', validate='one_to_one',
).merge(
    content_selection[[
        'Dataset', 'Task', 'SelectedDifficultyTolerance'
    ]],
    on=['Dataset', 'Task'], how='left', validate='one_to_one',
)
hyperparameter_selection['SelectionMetric'] = 'attempted_all_supported_NDCGAt10'
display(sequential_selection)
display(sequential_tuning_metrics.sort_values([
    'Dataset', 'Task', 'TransitionShrinkage', 'RecencyDecay', 'Segment'
]))
globals().pop('selected_sequential_recommendations', None)
_ = gc.collect()


,Dataset,Task,SelectedTransitionShrinkage,SelectedRecencyDecay,SelectionMetric,SelectionK,SelectionRelevance,SelectionCandidatePolicy,DiscoveryTuningLearners,ValidationLearnersUntouched
0,ASSISTments,problem,0.0,0.5,NDCGAtK,10,attempted,all_supported,5334,6667
1,ASSISTments,skill,10.0,0.5,NDCGAtK,10,attempted,all_supported,5334,6667
2,KDD,problem,0.0,0.5,NDCGAtK,10,attempted,all_supported,91,113
3,KDD,skill,0.0,0.9,NDCGAtK,10,attempted,all_supported,91,113


,Dataset,Task,CandidatePolicy,RelevanceDefinition,Model,Segment,K,PrecisionAtK,RecallAtK,NDCGAtK,...,CatalogCoverageAtK,MeanRecommendations,EvaluatedLearners,EvaluableRate,EvaluationScope,RecommendationMemoryBytes,MetricRuntimeSeconds,TransitionShrinkage,RecencyDecay,TuningSource
0,ASSISTments,problem,all_supported,attempted,sequential_transition_tuning,all,10,0.154407,0.109550,0.214191,...,0.345635,10.0,4799,0.899700,provided_learner_ids,20810377,1.254242,0.0,0.5,discovery_only
1,ASSISTments,problem,all_supported,attempted,sequential_transition_tuning,cold_start,10,0.000000,0.000000,0.000000,...,0.000251,10.0,13,0.899700,provided_learner_ids,20810377,1.254242,0.0,0.5,discovery_only
2,ASSISTments,problem,all_supported,attempted,sequential_transition_tuning,non_cold_start,10,0.154827,0.109847,0.214773,...,0.345635,10.0,4786,0.899700,provided_learner_ids,20810377,1.254242,0.0,0.5,discovery_only
3,ASSISTments,problem,all_supported,attempted,sequential_transition_tuning,all,10,0.153053,0.108825,0.210880,...,0.345735,10.0,4799,0.899700,provided_learner_ids,20810315,1.902353,0.0,0.7,discovery_only
4,ASSISTments,problem,all_supported,attempted,sequential_transition_tuning,cold_start,10,0.000000,0.000000,0.000000,...,0.000251,10.0,13,0.899700,provided_learner_ids,20810315,1.902353,0.0,0.7,discovery_only
5,ASSISTments,problem,all_supported,attempted,sequential_transition_tuning,non_cold_start,10,0.153468,0.109120,0.211452,...,0.345735,10.0,4786,0.899700,provided_learner_ids,20810315,1.902353,0.0,0.7,discovery_only
6,ASSISTments,problem,all_supported,attempted,sequential_transition_tuning,all,10,0.151740,0.107737,0.205935,...,0.346163,10.0,4799,0.899700,provided_learner_ids,20810357,1.209635,0.0,0.9,discovery_only
7,ASSISTments,problem,all_supported,attempted,sequential_transition_tuning,cold_start,10,0.000000,0.000000,0.000000,...,0.000251,10.0,13,0.899700,provided_learner_ids,20810357,1.209635,0.0,0.9,discovery_only
8,ASSISTments,problem,all_supported,attempted,sequential_transition_tuning,non_cold_start,10,0.152152,0.108030,0.206494,...,0.346163,10.0,4786,0.899700,provided_learner_ids,20810357,1.209635,0.0,0.9,discovery_only
9,ASSISTments,problem,all_supported,attempted,sequential_transition_tuning,all,10,0.154365,0.109468,0.213740,...,0.344302,10.0,4799,0.899700,provided_learner_ids,20810201,1.215948,10.0,0.5,discovery_only


## Step 14 — Consolidation and leakage assertions

Combines popularity, weak-skill and cluster staged outputs with the discovery-selected content, unrestricted-CF and sequential results. Large recommendation payloads are streamed into one Parquet file while model-independent and model-specific leakage/candidate/ranking assertions are applied.


In [26]:
import pyarrow as pa

assert transition_probabilities['transition_source'].eq(
    'discovery_early_adjacent_events_only'
).all()
for large_name in [
    'transition_probabilities', 'discovery_recent_items',
    'validation_recent_items',
]:
    globals().pop(large_name, None)
gc.collect()

selected_content_metrics = pd.read_csv(step13_model_paths['content_metrics'])
selected_neighbor_metrics = pd.read_csv(step13_model_paths['neighbor_metrics'])
selected_sequential_metrics = pd.read_csv(step13_model_paths['sequential_metrics'])
metrics_6_7 = pd.read_csv(OUTPUT_ROOT / 'steps_6_7_metrics.csv')
metrics_8_10 = pd.read_csv(OUTPUT_ROOT / 'steps_8_10_metrics.csv')
recommendation_metrics = pd.concat([
    metrics_6_7, metrics_8_10, selected_content_metrics, selected_neighbor_metrics,
    selected_sequential_metrics,
], ignore_index=True, sort=False)
required_recommendation_columns = {
    'Dataset', 'Task', 'Model', 'CandidatePolicy',
    'RelevanceDefinition', 'learner_id', 'item_id', 'score', 'rank',
}
recommendation_key = [
    'Dataset', 'Task', 'Model', 'CandidatePolicy', 'RelevanceDefinition',
    'learner_id', 'item_id',
]
final_recommendation_path = OUTPUT_ROOT / 'validation_recommendations.parquet'
recommendation_sources = [
    (OUTPUT_ROOT / 'steps_6_7_recommendations.parquet', set()),
    (OUTPUT_ROOT / 'steps_8_10_recommendations.parquet', set()),
    (step13_model_paths['content_recommendations'], set()),
    (step13_model_paths['neighbor_recommendations'], set()),
    (step13_model_paths['sequential_recommendations'], set()),
]
final_recommendation_columns = [
    'Dataset', 'Task', 'Model', 'CandidatePolicy', 'RelevanceDefinition',
    'learner_id', 'item_id', 'score', 'rank', 'score_source',
    'NeighborCount', 'NeighborWeighting', 'DifficultyTolerance',
    'TransitionShrinkage', 'RecencyDecay', 'ClusterEvidenceUse',
]
string_columns = {
    'Dataset', 'Task', 'Model', 'CandidatePolicy', 'RelevanceDefinition',
    'learner_id', 'item_id', 'score_source', 'NeighborWeighting',
    'ClusterEvidenceUse',
}
final_recommendation_schema = pa.schema([
    pa.field(column, (
        pa.string() if column in string_columns
        else (pa.int32() if column == 'rank' else pa.float64())
    ))
    for column in final_recommendation_columns
])
writer = pq.ParquetWriter(
    final_recommendation_path, final_recommendation_schema, compression='snappy'
)
validation_recommendation_rows = 0
try:
    for source_path, excluded_models in recommendation_sources:
        for batch in pq.ParquetFile(source_path).iter_batches(batch_size=100_000):
            frame = batch.to_pandas()
            if excluded_models:
                frame = frame[~frame['Model'].isin(excluded_models)].copy()
            if frame.empty:
                continue
            assert required_recommendation_columns.issubset(frame.columns)
            assert np.isfinite(pd.to_numeric(frame['score'], errors='coerce')).all()
            assert pd.to_numeric(frame['rank']).between(1, MAX_RECOMMENDATIONS).all()
            assert not frame.duplicated(recommendation_key).any()
            normalised = pd.DataFrame(index=frame.index)
            for column in final_recommendation_columns:
                if column in string_columns:
                    normalised[column] = (
                        frame[column].astype('string')
                        if column in frame.columns else pd.Series(
                            pd.NA, index=frame.index, dtype='string'
                        )
                    )
                elif column == 'rank':
                    normalised[column] = pd.to_numeric(frame[column]).astype('int32')
                else:
                    normalised[column] = (
                        pd.to_numeric(frame[column], errors='coerce').astype(float)
                        if column in frame.columns else np.nan
                    )
            table = pa.Table.from_pandas(
                normalised, schema=final_recommendation_schema,
                preserve_index=False, safe=False,
            )
            writer.write_table(table)
            validation_recommendation_rows += len(normalised)
            del frame, normalised, table
finally:
    writer.close()
assert validation_recommendation_rows > 0
assert pq.ParquetFile(final_recommendation_path).metadata.num_rows == (
    validation_recommendation_rows
)
assert recommendation_metrics['K'].le(MAX_RECOMMENDATIONS).all()
assert recommendation_metrics['EvaluationScope'].eq('validation').all()

leakage_audit_rows = []
for task_definition in TASK_DEFINITIONS:
    dataset = task_definition['Dataset']
    task = task_definition['Task']
    tables = load_task_tables(task_definition)
    discovery_ids = set(tables['learners'].loc[
        tables['learners']['cohort'].eq('discovery'), 'learner_id'
    ].astype(str))
    validation_ids = set(tables['learners'].loc[
        tables['learners']['cohort'].eq('validation'), 'learner_id'
    ].astype(str))
    catalog_ids = set(tables['catalog']['item_id'].astype(str))
    assert discovery_ids.isdisjoint(validation_ids)
    assert len(catalog_ids) >= MAX_RECOMMENDATIONS
    seen = tables['early_history'][
        tables['early_history']['in_candidate_catalog']
    ][['learner_id', 'item_id']].drop_duplicates()
    seen['learner_id'] = seen['learner_id'].astype(str)
    seen['item_id'] = seen['item_id'].astype(str)
    task_rows_checked = 0
    audit_columns = [
        'Dataset', 'Task', 'CandidatePolicy', 'learner_id', 'item_id'
    ]
    for batch in pq.ParquetFile(final_recommendation_path).iter_batches(
        batch_size=100_000, columns=audit_columns
    ):
        frame = batch.to_pandas()
        frame = frame[
            frame['Dataset'].eq(dataset) & frame['Task'].eq(task)
        ].copy()
        if frame.empty:
            continue
        assert set(frame['learner_id'].astype(str)).issubset(validation_ids)
        assert set(frame['item_id'].astype(str)).issubset(catalog_ids)
        novel = frame[frame['CandidatePolicy'].eq('novel_only')].copy()
        if not novel.empty:
            novel['learner_id'] = novel['learner_id'].astype(str)
            novel['item_id'] = novel['item_id'].astype(str)
            overlap = novel.merge(
                seen, on=['learner_id', 'item_id'], how='inner'
            )
            assert overlap.empty
        task_rows_checked += len(frame)
        del frame, novel
    assert task_rows_checked > 0
    leakage_audit_rows.append({
        'Dataset': dataset, 'Task': task,
        'DiscoveryValidationDisjoint': True,
        'RecommendationsValidationOnly': True,
        'CandidatesInFrozenCatalog': True,
        'NovelRecommendationsExcludeSeen': True,
        'CatalogSupportsMaximumK': True,
        'RecommendationRowsChecked': task_rows_checked,
    })
    del tables, seen
    gc.collect()

assert recommendation_metrics.loc[
    recommendation_metrics['Model'].eq('learner_neighbor_cf'),
    'TrainingLearnerCohort'
].eq('discovery').all()
assert recommendation_metrics.loc[
    recommendation_metrics['Dataset'].eq('KDD')
    & recommendation_metrics['Model'].isin([
        'same_cluster_neighbor_cf', 'cluster_popularity'
    ]), 'ClusterEvidenceUse'
].eq('exploratory_ablation_only').all()
leakage_audit = pd.DataFrame(leakage_audit_rows)
display(leakage_audit)


,Dataset,Task,DiscoveryValidationDisjoint,RecommendationsValidationOnly,CandidatesInFrozenCatalog,NovelRecommendationsExcludeSeen,CatalogSupportsMaximumK,RecommendationRowsChecked
0,ASSISTments,problem,True,True,True,True,True,3378581
1,ASSISTments,skill,True,True,True,True,True,2627618
2,KDD,problem,True,True,True,True,True,63172
3,KDD,skill,True,True,True,True,True,56562


## Steps 15-16 — Final artifacts and decision rules

Creates the final comparison at the predeclared primary slice (attempted relevance, all-supported candidates, all evaluable learners, K=10), applies coverage-aware rules relative to discovery-early popularity, records cluster-versus-unrestricted-CF decisions, and writes the consolidated Phase 3 package. KDD decisions are always secondary evidence.


In [27]:
comparison_slice = recommendation_metrics[
    recommendation_metrics['RelevanceDefinition'].eq('attempted')
    & recommendation_metrics['CandidatePolicy'].eq('all_supported')
    & recommendation_metrics['Segment'].eq('all')
    & recommendation_metrics['K'].eq(10)
].copy()
baseline = comparison_slice[
    comparison_slice['Model'].eq('popularity_discovery_early')
][['Dataset', 'Task', 'RecallAtK', 'NDCGAtK', 'CatalogCoverageAtK']].rename(columns={
    'RecallAtK': 'PopularityRecallAtK',
    'NDCGAtK': 'PopularityNDCGAtK',
    'CatalogCoverageAtK': 'PopularityCatalogCoverageAtK',
})
model_comparison = comparison_slice.merge(
    baseline, on=['Dataset', 'Task'], how='left', validate='many_to_one'
)
model_comparison['RecallDeltaVsPopularity'] = (
    model_comparison['RecallAtK'] - model_comparison['PopularityRecallAtK']
)
model_comparison['NDCGDeltaVsPopularity'] = (
    model_comparison['NDCGAtK'] - model_comparison['PopularityNDCGAtK']
)
model_comparison['CoverageRatioVsPopularity'] = (
    model_comparison['CatalogCoverageAtK'].div(
        model_comparison['PopularityCatalogCoverageAtK'].replace(0, np.nan)
    ).fillna(0.0)
)
model_comparison['BeatsPopularity'] = (
    model_comparison['RecallDeltaVsPopularity'].gt(0)
    & model_comparison['NDCGDeltaVsPopularity'].gt(0)
    & model_comparison['CoverageRatioVsPopularity'].ge(0.50)
)

ordinary_cf = model_comparison[
    model_comparison['Model'].eq('learner_neighbor_cf')
][['Dataset', 'Task', 'RecallAtK', 'NDCGAtK', 'CatalogCoverageAtK']].rename(columns={
    'RecallAtK': 'OrdinaryCFRecallAtK',
    'NDCGAtK': 'OrdinaryCFNDCGAtK',
    'CatalogCoverageAtK': 'OrdinaryCFCatalogCoverageAtK',
})
model_comparison = model_comparison.merge(
    ordinary_cf, on=['Dataset', 'Task'], how='left', validate='many_to_one'
)
model_comparison['BeatsOrdinaryCF'] = (
    model_comparison['RecallAtK'].gt(model_comparison['OrdinaryCFRecallAtK'])
    & model_comparison['NDCGAtK'].gt(model_comparison['OrdinaryCFNDCGAtK'])
    & model_comparison['CatalogCoverageAtK'].ge(
        0.50 * model_comparison['OrdinaryCFCatalogCoverageAtK']
    )
)

def phase3_decision(row):
    if row['Model'] == 'popularity_discovery_early':
        return 'reference_baseline'
    if row['Dataset'] == 'KDD':
        return 'secondary_evidence_only'
    if row['Model'] == 'popularity_discovery_future':
        return 'supervised_popularity_comparator'
    if row['Model'] in {'same_cluster_neighbor_cf', 'cluster_popularity'}:
        return (
            'eligible_for_phase4' if bool(row['BeatsOrdinaryCF'])
            else 'exclude_cluster_signal'
        )
    if row['Model'] == 'sequential_transition' and not bool(row['BeatsPopularity']):
        return 'interpretable_comparison_only'
    return (
        'eligible_for_phase4' if bool(row['BeatsPopularity'])
        else 'does_not_pass_phase3_gate'
    )

model_comparison['Decision'] = model_comparison.apply(phase3_decision, axis=1)
model_comparison['DecisionScope'] = np.where(
    model_comparison['Dataset'].eq('ASSISTments')
    & model_comparison['Task'].eq('problem'),
    'primary', 'secondary',
)
recommendation_coverage = recommendation_metrics[[
    'Dataset', 'Task', 'CandidatePolicy', 'RelevanceDefinition', 'Model',
    'Segment', 'K', 'CatalogCoverageAtK', 'MeanRecommendations',
    'EvaluatedLearners', 'EvaluableRate',
]].copy()
display(model_comparison[[
    'Dataset', 'Task', 'Model', 'RecallAtK', 'NDCGAtK',
    'CatalogCoverageAtK', 'RecallDeltaVsPopularity',
    'NDCGDeltaVsPopularity', 'CoverageRatioVsPopularity', 'Decision',
]].sort_values(['Dataset', 'Task', 'NDCGAtK'], ascending=[True, True, False]))


,Dataset,Task,Model,RecallAtK,NDCGAtK,CatalogCoverageAtK,RecallDeltaVsPopularity,NDCGDeltaVsPopularity,CoverageRatioVsPopularity,Decision
20,ASSISTments,problem,learner_neighbor_cf,0.270136,0.498392,0.219513,0.262903,0.486491,873.2,eligible_for_phase4
10,ASSISTments,problem,same_cluster_neighbor_cf,0.263530,0.488105,0.227256,0.256296,0.476205,904.0,exclude_cluster_signal
24,ASSISTments,problem,sequential_transition,0.116604,0.219903,0.381005,0.109370,0.208003,1515.6,eligible_for_phase4
11,ASSISTments,problem,cluster_popularity,0.017859,0.031371,0.000503,0.010625,0.019470,2.0,exclude_cluster_signal
1,ASSISTments,problem,popularity_discovery_future,0.003712,0.031343,0.000251,-0.003522,0.019442,1.0,supervised_popularity_comparator
0,ASSISTments,problem,popularity_discovery_early,0.007234,0.011900,0.000251,0.000000,0.000000,1.0,reference_baseline
18,ASSISTments,problem,content_problem,0.003517,0.007825,0.176978,-0.003717,-0.004075,704.0,does_not_pass_phase3_gate
21,ASSISTments,skill,learner_neighbor_cf,0.689105,0.734290,0.975155,0.428747,0.478706,15.7,eligible_for_phase4
12,ASSISTments,skill,same_cluster_neighbor_cf,0.668541,0.713161,0.962733,0.408183,0.457576,15.5,exclude_cluster_signal
25,ASSISTments,skill,sequential_transition,0.432834,0.489179,0.975155,0.172476,0.233595,15.7,eligible_for_phase4


In [28]:
import shutil

hyperparameter_tuning_metrics = pd.concat([
    neighbor_tuning_metrics, neighbor_weighting_tuning_metrics,
    content_tuning_metrics, sequential_tuning_metrics,
], ignore_index=True, sort=False)
final_paths = {
    'metrics': OUTPUT_ROOT / 'recommendation_metrics.csv',
    'coverage': OUTPUT_ROOT / 'recommendation_coverage.csv',
    'comparison': OUTPUT_ROOT / 'model_comparison.csv',
    'recommendations': OUTPUT_ROOT / 'validation_recommendations.parquet',
    'popularity_scores': OUTPUT_ROOT / 'popularity_scores.parquet',
    'content_scores': OUTPUT_ROOT / 'content_scores.parquet',
    'neighbor_config': OUTPUT_ROOT / 'neighbor_configuration.csv',
    'transition_probabilities': OUTPUT_ROOT / 'transition_probabilities.parquet',
    'hyperparameter_tuning': OUTPUT_ROOT / 'hyperparameter_tuning_metrics.csv',
    'hyperparameter_selection': OUTPUT_ROOT / 'hyperparameter_selection.csv',
    'leakage_audit': OUTPUT_ROOT / 'leakage_audit.csv',
    'config': OUTPUT_ROOT / 'phase3_config.json',
}
recommendation_metrics.to_csv(final_paths['metrics'], index=False)
recommendation_coverage.to_csv(final_paths['coverage'], index=False)
model_comparison.to_csv(final_paths['comparison'], index=False)
assert final_paths['recommendations'] == final_recommendation_path
shutil.copyfile(
    OUTPUT_ROOT / 'step6_popularity_scores.parquet',
    final_paths['popularity_scores'],
)
shutil.copyfile(
    step13_model_paths['content_scores'], final_paths['content_scores']
)
selected_neighbor_configuration.to_csv(final_paths['neighbor_config'], index=False)
shutil.copyfile(
    steps_11_12_paths['transition_probabilities'],
    final_paths['transition_probabilities'],
)
hyperparameter_tuning_metrics.to_csv(
    final_paths['hyperparameter_tuning'], index=False
)
hyperparameter_selection.to_csv(final_paths['hyperparameter_selection'], index=False)
leakage_audit.to_csv(final_paths['leakage_audit'], index=False)
phase3_config = {
    'phase': 3, 'implemented_steps': list(range(1, 17)),
    'selection_source': 'discovery_only',
    'execution_graph': 'single_validation_evaluation_after_discovery_selection',
    'final_evaluation_cohort': 'validation',
    'primary_decision_slice': {
        'dataset': 'ASSISTments', 'relevance': 'attempted',
        'candidate_policy': 'all_supported', 'segment': 'all', 'k': 10,
    },
    'coverage_floor_ratio': 0.50,
    'candidate_policies': list(CANDIDATE_POLICIES),
    'relevance_definitions': RELEVANCE_DEFINITIONS,
    'metric_ks': list(METRIC_KS), 'random_state': RANDOM_STATE,
    'validation_future_role': 'evaluation_only',
    'kdd_cluster_evidence': 'exploratory_ablation_only',
    'dense_learner_item_matrix_constructed': False,
}
with open(final_paths['config'], 'w', encoding='utf-8') as file:
    json.dump(phase3_config, file, indent=2)

artifact_manifest = pd.DataFrame([
    {
        'Artifact': name, 'File': path.name,
        'Rows': (
            pq.ParquetFile(path).metadata.num_rows
            if path.suffix == '.parquet'
            else (len(pd.read_csv(path)) if path.suffix == '.csv' else 1)
        ),
        'Bytes': path.stat().st_size,
    }
    for name, path in final_paths.items()
])
artifact_manifest.to_csv(OUTPUT_ROOT / 'artifact_manifest.csv', index=False)
assert len(pd.read_csv(final_paths['metrics'])) == len(recommendation_metrics)
assert pq.ParquetFile(
    final_paths['recommendations']
).metadata.num_rows == validation_recommendation_rows
assert len(artifact_manifest) == len(final_paths)
display(artifact_manifest)


,Artifact,File,Rows,Bytes
0,metrics,recommendation_metrics.csv,840,235222
1,coverage,recommendation_coverage.csv,840,103347
2,comparison,model_comparison.csv,28,14468
3,recommendations,validation_recommendations.parquet,6125933,22956506
4,popularity_scores,popularity_scores.parquet,163576,1193533
5,content_scores,content_scores.parquet,6770282,145442689
6,neighbor_config,neighbor_configuration.csv,4,500
7,transition_probabilities,transition_probabilities.parquet,802669,6621768
8,hyperparameter_tuning,hyperparameter_tuning_metrics.csv,125,36764
9,hyperparameter_selection,hyperparameter_selection.csv,4,557
